In [3]:
"""
Part A - Data Foundation
Tata Play Subscriber Retention & Support Copilot

Tasks:
1. Audit raw datasets
2. Clean subscribers
3. Clean recharges
4. Save clean CSV files
5. Create SQLite database
6. Execute four SQL queries
"""

import os
import sqlite3
import pandas as pd
from google.colab import files
uploaded = files.upload()
import zipfile
zip_file = next(name for name in uploaded if name.endswith(".zip"))

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall("/content")

print("Files extracted successfully!")
import os

for root, dirs, files_list in os.walk("/content"):
    if "subscribers.csv" in files_list:
        print(os.path.join(root, "subscribers.csv"))

# --------------------------------------------------
# 1. FILE PATHS
# --------------------------------------------------

RAW_DIR = "data/raw"
CLEAN_DIR = "data/clean"
OUTPUT_DIR = "outputs"
SQL_DIR = "sql"

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SQL_DIR, exist_ok=True)

SUBSCRIBERS_FILE = os.path.join(RAW_DIR, "subscribers.csv")
RECHARGES_FILE = os.path.join(RAW_DIR, "recharges.csv")
PLANS_FILE = os.path.join(RAW_DIR, "plans.csv")

DB_FILE = "data/tataplay.db"


# --------------------------------------------------
# 2. LOAD RAW DATA
# --------------------------------------------------

print("\n========== PART A: DATA FOUNDATION ==========")

subscribers = pd.read_csv(SUBSCRIBERS_FILE)
recharges = pd.read_csv(RECHARGES_FILE)
plans = pd.read_csv(PLANS_FILE)

print("\nRaw datasets loaded successfully.")


# --------------------------------------------------
# 3. AUDIT RAW DATA
# --------------------------------------------------

def audit_dataset(df, name):
    print(f"\n========== {name.upper()} AUDIT ==========")

    print("\nShape:")
    print(df.shape)

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    missing = pd.DataFrame({
        "Missing_Count": df.isna().sum(),
        "Missing_Percentage": (
            df.isna().sum() / len(df) * 100
        ).round(2)
    })

    print(missing)

    print("\nExact duplicate rows:")
    print(df.duplicated().sum())


audit_dataset(subscribers, "Subscribers")
audit_dataset(recharges, "Recharges")


# Additional subscriber audit figures

print("\n========== SUBSCRIBER AUDIT CHECKS ==========")

print("Total subscriber rows:", len(subscribers))
print("Distinct subscriber IDs:", subscribers["subscriber_id"].nunique())
print("Duplicate subscriber rows:", subscribers.duplicated().sum())

print(
    "Blank age:",
    subscribers["age"].isna().sum()
)

age_numeric = pd.to_numeric(
    subscribers["age"], errors="coerce"
)

print(
    "Age values outside 10-100:",
    ((age_numeric < 10) | (age_numeric > 100)).sum()
)

print(
    "Blank payment mode:",
    subscribers["payment_mode"].isna().sum()
)

print(
    "Distinct raw city spellings:",
    subscribers["city"].nunique()
)

print(
    "Distinct raw auto-renew spellings:",
    subscribers["auto_renew"].nunique()
)

print(
    "Blank next-cycle recharge:",
    subscribers["next_cycle_recharge_inr"].isna().sum()
)


# Additional recharge audit figures

print("\n========== RECHARGE AUDIT CHECKS ==========")

print("Total recharge rows:", len(recharges))
print("Duplicate recharge rows:", recharges.duplicated().sum())
print("Distinct recharge IDs:", recharges["recharge_id"].nunique())

print(
    "Distinct raw status spellings:",
    recharges["status"].nunique()
)

amount_text = recharges["amount_inr"].astype(str).str.strip()

not_plain_number = pd.to_numeric(
    amount_text, errors="coerce"
).isna()

print(
    "Amounts that are not plain numbers:",
    not_plain_number.sum()
)

print(
    "Recharge rows with unknown subscriber IDs:",
    (~recharges["subscriber_id"].isin(
        subscribers["subscriber_id"]
    )).sum()
)


# --------------------------------------------------
# 4. CLEAN SUBSCRIBERS
# --------------------------------------------------

print("\n========== CLEANING SUBSCRIBERS ==========")

# Remove exact duplicate rows
subscribers = subscribers.drop_duplicates().copy()

print("Rows after removing duplicates:", len(subscribers))


# Standardise city names

city_mapping = {
    "chennai": "Chennai",
    "madras": "Chennai",
    "mumbai": "Mumbai",
    "bombay": "Mumbai",
    "bengaluru": "Bengaluru",
    "bangalore": "Bengaluru",
    "delhi": "Delhi",
    "new delhi": "Delhi",
    "hyderabad": "Hyderabad",
    "hyd": "Hyderabad",
    "pune": "Pune",
    "kolkata": "Kolkata",
    "calcutta": "Kolkata"
}

subscribers["city"] = (
    subscribers["city"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(city_mapping)
)


# Standardise auto-renew values

subscribers["auto_renew"] = (
    subscribers["auto_renew"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "yes": "Yes",
        "y": "Yes",
        "no": "No",
        "n": "No"
    })
)


# Convert age to numeric

subscribers["age"] = pd.to_numeric(
    subscribers["age"],
    errors="coerce"
)

# Replace impossible ages with missing values
invalid_age = (
    (subscribers["age"] < 10) |
    (subscribers["age"] > 100)
)

subscribers.loc[invalid_age, "age"] = pd.NA


# Convert numeric columns to numeric types

numeric_columns = [
    "tenure_months",
    "add_on_count",
    "monthly_recharge_inr",
    "hours_watched_per_week",
    "hd_hours_per_week",
    "sports_hours_per_week",
    "channels_watched",
    "support_calls_90d",
    "churned"
]

for column in numeric_columns:
    subscribers[column] = pd.to_numeric(
        subscribers[column],
        errors="coerce"
    )


# Display cleaned subscriber results

print("\nCity distribution:")
print(subscribers["city"].value_counts())

print("\nAuto-renew distribution:")
print(subscribers["auto_renew"].value_counts())

print("\nMissing age after cleaning:")
print(subscribers["age"].isna().sum())

print("\nMissing payment mode:")
print(subscribers["payment_mode"].isna().sum())

print("\nChurn distribution:")
print(subscribers["churned"].value_counts())

churn_percentage = (
    subscribers["churned"].mean() * 100
)

print(f"Churn percentage: {churn_percentage:.2f}%")


# --------------------------------------------------
# 5. CLEAN RECHARGES
# --------------------------------------------------

print("\n========== CLEANING RECHARGES ==========")

# Remove exact duplicate rows
recharges = recharges.drop_duplicates().copy()

print("After removing duplicates:", len(recharges))


# Remove recharge rows for unknown subscribers

valid_subscriber_ids = set(
    subscribers["subscriber_id"]
)

recharges = recharges[
    recharges["subscriber_id"].isin(valid_subscriber_ids)
].copy()

print("After removing unknown subscribers:", len(recharges))


# Standardise recharge status

recharges["status"] = (
    recharges["status"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# Convert recharge amounts to numbers.
# Values written with an INR prefix are rebuilt from
# the subscriber's monthly recharge amount, as required.

raw_amount = recharges["amount_inr"].astype("string").str.strip()

has_inr_prefix = raw_amount.str.match(
    r"(?i)^INR\s*",
    na=False
)

amount_numeric = pd.to_numeric(
    raw_amount,
    errors="coerce"
)

# INR-prefixed text is not treated as a plain numeric amount.
amount_numeric.loc[has_inr_prefix] = pd.NA

# Also rebuild any other values that cannot be converted.
amount_numeric = amount_numeric.astype("float64")

recharges["amount_inr"] = amount_numeric


# Get each subscriber's monthly recharge amount

monthly_amounts = subscribers.set_index(
    "subscriber_id"
)["monthly_recharge_inr"]


# Rebuild invalid recharge amounts

rebuild_mask = recharges["amount_inr"].isna()

recharges.loc[rebuild_mask, "amount_inr"] = (
    recharges.loc[rebuild_mask, "subscriber_id"]
    .map(monthly_amounts)
)

print(
    "Recharge amounts rebuilt:",
    int(rebuild_mask.sum())
)


# Parse recharge dates

recharges["recharge_date"] = pd.to_datetime(
    recharges["recharge_date"],
    errors="coerce"
)

# Store dates in ISO format in the clean CSV
recharges["recharge_date"] = (
    recharges["recharge_date"].dt.strftime("%Y-%m-%d")
)


# Display recharge cleaning results

print("\nRecharge status counts:")
print(recharges["status"].value_counts())

successful_recharges = recharges[
    recharges["status"] == "success"
]

print(
    "\nTotal successful recharge amount:",
    successful_recharges["amount_inr"].sum()
)


# --------------------------------------------------
# 6. SAVE CLEAN CSV FILES
# --------------------------------------------------

print("\n========== SAVING CLEAN DATA ==========")

subscribers.to_csv(
    os.path.join(CLEAN_DIR, "subscribers.csv"),
    index=False
)

recharges.to_csv(
    os.path.join(CLEAN_DIR, "recharges.csv"),
    index=False
)

plans.to_csv(
    os.path.join(CLEAN_DIR, "plans.csv"),
    index=False
)

print("Clean CSV files saved successfully.")

print("Clean subscribers:", len(subscribers))
print("Clean recharges:", len(recharges))
print("Clean plans:", len(plans))


# --------------------------------------------------
# 7. CREATE SQLITE DATABASE
# --------------------------------------------------

print("\n========== CREATING SQLITE DATABASE ==========")

os.makedirs("data", exist_ok=True)

connection = sqlite3.connect(DB_FILE)

subscribers.to_sql(
    "subscribers",
    connection,
    if_exists="replace",
    index=False
)

recharges.to_sql(
    "recharges",
    connection,
    if_exists="replace",
    index=False
)

plans.to_sql(
    "plans",
    connection,
    if_exists="replace",
    index=False
)

print("SQLite database created:", DB_FILE)


# --------------------------------------------------
# 8. SQL QUERIES
# --------------------------------------------------

queries = {

    # Q1: Revenue by plan

    "Q1 Revenue by Plan": """
        SELECT
            p.plan_name,
            SUM(r.amount_inr) AS total_revenue,
            ROUND(AVG(r.amount_inr), 2) AS average_revenue,
            COUNT(DISTINCT s.subscriber_id) AS subscriber_count

        FROM recharges r

        INNER JOIN subscribers s
            ON r.subscriber_id = s.subscriber_id

        INNER JOIN plans p
            ON s.plan_id = p.plan_id

        WHERE LOWER(r.status) = 'success'

        GROUP BY p.plan_name

        ORDER BY total_revenue DESC;
    """,

    # Q2: Subscribers who never successfully recharged

    "Q2 Never Successfully Recharged": """
        SELECT
            s.subscriber_id,
            s.city,
            s.tenure_months

        FROM subscribers s

        LEFT JOIN recharges r
            ON s.subscriber_id = r.subscriber_id
            AND LOWER(r.status) = 'success'

        WHERE r.recharge_id IS NULL

        ORDER BY s.subscriber_id;
    """,

    # Q3: Ranking within each city

    "Q3 Ranking Within City": """
        WITH subscriber_revenue AS (
            SELECT
                s.subscriber_id,
                s.city,
                SUM(r.amount_inr) AS total_success

            FROM subscribers s

            INNER JOIN recharges r
                ON s.subscriber_id = r.subscriber_id

            WHERE LOWER(r.status) = 'success'

            GROUP BY s.subscriber_id, s.city
        ),

        ranked AS (
            SELECT
                subscriber_id,
                city,
                total_success,

                RANK() OVER (
                    PARTITION BY city
                    ORDER BY total_success DESC
                ) AS rank_value,

                DENSE_RANK() OVER (
                    PARTITION BY city
                    ORDER BY total_success DESC
                ) AS dense_rank_value,

                ROW_NUMBER() OVER (
                    PARTITION BY city
                    ORDER BY total_success DESC, subscriber_id
                ) AS row_number_value

            FROM subscriber_revenue
        )

        SELECT *
        FROM ranked
        ORDER BY city, row_number_value;
    """,

    # Q4: Running total for TP00010

    "Q4 Running Total": """
        SELECT
            subscriber_id,
            recharge_id,
            recharge_date,
            amount_inr,

            SUM(amount_inr) OVER (
                ORDER BY recharge_date, recharge_id
                ROWS BETWEEN UNBOUNDED PRECEDING
                AND CURRENT ROW
            ) AS running_total

        FROM recharges

        WHERE subscriber_id = 'TP00010'
          AND LOWER(status) = 'success'

        ORDER BY recharge_date, recharge_id;
    """
}


# --------------------------------------------------
# 9. SAVE SQL QUERIES TO FILE
# --------------------------------------------------

sql_file = os.path.join(SQL_DIR, "queries.sql")

with open(sql_file, "w", encoding="utf-8") as file:
    for query_name, query in queries.items():
        file.write(f"\n-- {query_name}\n")
        file.write(query)
        file.write("\n")

print("\nSQL queries saved to:", sql_file)


# --------------------------------------------------
# 10. EXECUTE SQL QUERIES
# --------------------------------------------------

print("\n========== EXECUTING SQL QUERIES ==========")

for query_name, query in queries.items():

    print(f"\n========== {query_name} ==========")

    result = pd.read_sql_query(
        query,
        connection
    )

    print(result.to_string(index=False))

    # Save individual results for inspection
    filename = (
        query_name.lower()
        .replace(" ", "_")
        .replace(":", "")
        + ".csv"
    )

    result.to_csv(
        os.path.join(OUTPUT_DIR, filename),
        index=False
    )


# --------------------------------------------------
# 11. Q1 CROSS-CHECK USING PANDAS GROUPBY
# --------------------------------------------------

print("\n========== PANDAS GROUPBY CROSS-CHECK ==========")

successful = recharges[
    recharges["status"] == "success"
].copy()

pandas_check = (
    successful
    .merge(
        subscribers[["subscriber_id", "plan_id"]],
        on="subscriber_id",
        how="inner"
    )
    .merge(
        plans[["plan_id", "plan_name"]],
        on="plan_id",
        how="inner"
    )
    .groupby("plan_name")
    .agg(
        total_revenue=("amount_inr", "sum"),
        average_revenue=("amount_inr", "mean"),
        subscriber_count=("subscriber_id", "nunique")
    )
    .reset_index()
)

pandas_check["average_revenue"] = (
    pandas_check["average_revenue"].round(2)
)

pandas_check = pandas_check.sort_values(
    "total_revenue",
    ascending=False
)

print(pandas_check.to_string(index=False))


# --------------------------------------------------
# 12. Q3 RANK = 1 COUNT PER CITY
# --------------------------------------------------

print("\n========== RANK 1 COUNT PER CITY ==========")

ranked_result = pd.read_sql_query(
    queries["Q3 Ranking Within City"],
    connection
)

rank_one_counts = (
    ranked_result[
        ranked_result["rank_value"] == 1
    ]
    .groupby("city")
    .size()
    .reset_index(name="subscribers_with_rank_1")
)

print(rank_one_counts.to_string(index=False))


# Kolkata first four rows

print("\n========== KOLKATA TOP FOUR ==========")

kolkata_top_four = ranked_result[
    ranked_result["city"] == "Kolkata"
].head(4)

print(kolkata_top_four.to_string(index=False))


# --------------------------------------------------
# 13. CLOSE DATABASE
# --------------------------------------------------

connection.close()

print("\n========== PART A COMPLETED ==========")
print("Clean CSV files, SQLite database and SQL queries created.")

Saving tataplay_colab_dataset.zip to tataplay_colab_dataset (2).zip
Files extracted successfully!
/content/Tata-Play-Retention-Copilot/subscribers.csv
/content/data/raw/subscribers.csv
/content/data/clean/subscribers.csv

========== PART A: DATA FOUNDATION ==========

Raw datasets loaded successfully.

========== SUBSCRIBERS AUDIT ==========

Shape:
(2415, 17)

Data types:
subscriber_id               object
city                        object
plan_id                     object
tenure_months                int64
age                        float64
payment_mode                object
auto_renew                  object
add_on_count                 int64
monthly_recharge_inr         int64
hours_watched_per_week     float64
hd_hours_per_week          float64
sports_hours_per_week      float64
channels_watched             int64
support_calls_90d            int64
closure_request_logged      object
next_cycle_recharge_inr    float64
churned                      int64
dtype: object

Missing values

In [ ]:

from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. PATHS AND SETTINGS
# ---------------------------------------------------------
ROOT = Path("/content")
CLEAN_DIR = ROOT / "data" / "clean"
OUTPUT_DIR = ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CUTOFF_DATE = pd.Timestamp("2026-03-31")

NUMERIC_FEATURES = [
    "tenure_months",
    "age",
    "add_on_count",
    "monthly_recharge_inr",
    "hours_watched_per_week",
    "hd_hours_per_week",
    "sports_hours_per_week",
    "channels_watched",
    "support_calls_90d",
    "n_success_6m",
    "n_failed_6m",
    "recency_days",
]

CATEGORICAL_FEATURES = [
    "city",
    "plan_id",
    "payment_mode",
    "auto_renew",
]

LEAKAGE_FEATURES = [
    "closure_request_logged",
    "next_cycle_recharge_inr",
]

TARGET = "churned"


# ---------------------------------------------------------
# 2. LOAD CLEAN DATA
# ---------------------------------------------------------

subscribers = pd.read_csv(CLEAN_DIR / "subscribers.csv")
recharges = pd.read_csv(CLEAN_DIR / "recharges.csv")

print("\n========== PART B: CHURN PREDICTION ==========")
print("Subscribers:", subscribers.shape)
print("Recharges:", recharges.shape)

required_subscriber_columns = (
    NUMERIC_FEATURES[:9]
    + CATEGORICAL_FEATURES
    + [TARGET, "subscriber_id"]
)

missing_columns = [
    col for col in required_subscriber_columns
    if col not in subscribers.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in subscribers.csv: {missing_columns}"
    )

# Convert target into 0 and 1 if it is stored as text.
if subscribers[TARGET].dtype == "object":
    target_map = {
        "yes": 1,
        "true": 1,
        "1": 1,
        "churned": 1,
        "no": 0,
        "false": 0,
        "0": 0,
        "stayed": 0,
        "stay": 0,
    }

    subscribers[TARGET] = (
        subscribers[TARGET]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(target_map)
    )

subscribers[TARGET] = pd.to_numeric(
    subscribers[TARGET], errors="coerce"
)

if subscribers[TARGET].isna().any():
    raise ValueError("The churn target contains missing or unknown values.")

subscribers[TARGET] = subscribers[TARGET].astype(int)

if not set(subscribers[TARGET].unique()).issubset({0, 1}):
    raise ValueError("The churn target must contain only 0 and 1.")


# ---------------------------------------------------------
# 3. RECHARGE FEATURE ENGINEERING
# ---------------------------------------------------------

# Identify the recharge date column.
date_candidates = ["recharge_date", "date", "transaction_date"]
date_col = next(
    (col for col in date_candidates if col in recharges.columns),
    None
)

# Identify the recharge status column.
status_candidates = ["status", "recharge_status", "transaction_status"]
status_col = next(
    (col for col in status_candidates if col in recharges.columns),
    None
)

if date_col is None or status_col is None:
    raise ValueError(
        "Could not identify recharge date/status columns. "
        f"Available columns: {list(recharges.columns)}"
    )

if "subscriber_id" not in recharges.columns:
    raise ValueError("recharges.csv must contain subscriber_id.")

recharges[date_col] = pd.to_datetime(
    recharges[date_col], errors="coerce"
)

recharges[status_col] = (
    recharges[status_col]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Use only transactions available on or before the cutoff date.
eligible = recharges[
    recharges[date_col].notna()
    & (recharges[date_col] <= CUTOFF_DATE)
].copy()

six_month_start = CUTOFF_DATE - pd.DateOffset(months=6)

six_month_rows = eligible[
    eligible[date_col] > six_month_start
].copy()

successful_6m = six_month_rows[
    six_month_rows[status_col] == "success"
]

failed_6m = six_month_rows[
    six_month_rows[status_col] == "failed"
]

success_counts = (
    successful_6m.groupby("subscriber_id")
    .size()
    .rename("n_success_6m")
)

failed_counts = (
    failed_6m.groupby("subscriber_id")
    .size()
    .rename("n_failed_6m")
)

# Last successful recharge up to the cutoff date.
all_successful = eligible[
    eligible[status_col] == "success"
]

last_success = (
    all_successful.groupby("subscriber_id")[date_col]
    .max()
)

recency = (
    (CUTOFF_DATE - last_success)
    .dt.days
    .rename("recency_days")
)

recharge_features = pd.concat(
    [success_counts, failed_counts, recency],
    axis=1
).reset_index()

subscribers = subscribers.merge(
    recharge_features,
    on="subscriber_id",
    how="left"
)

# Subscribers with no recharge rows have zero counts.
subscribers["n_success_6m"] = (
    subscribers["n_success_6m"].fillna(0)
)

subscribers["n_failed_6m"] = (
    subscribers["n_failed_6m"].fillna(0)
)

# Recency remains missing when no successful recharge exists.
subscribers["n_success_6m"] = (
    subscribers["n_success_6m"].astype(int)
)

subscribers["n_failed_6m"] = (
    subscribers["n_failed_6m"].astype(int)
)

print("\nRecharge feature checks:")
print("Missing recency_days:", subscribers["recency_days"].isna().sum())
print("Total successful recharges in 6 months:",
      subscribers["n_success_6m"].sum())
print("Total failed recharges in 6 months:",
      subscribers["n_failed_6m"].sum())


# ---------------------------------------------------------
# 4. CLASS BALANCE AND MODELLING TABLE
# ---------------------------------------------------------

print("\n========== CLASS BALANCE ==========")

class_counts = subscribers[TARGET].value_counts().sort_index()
class_percentages = (
    subscribers[TARGET].value_counts(normalize=True).sort_index() * 100
)

print("Stayed (0):", class_counts.get(0, 0))
print("Churned (1):", class_counts.get(1, 0))
print("Churn percentage:", round(class_percentages.get(1, 0), 2), "%")

X = subscribers[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = subscribers[TARGET].copy()
subscriber_ids = subscribers["subscriber_id"].copy()


# ---------------------------------------------------------
# 5. TRAIN / VALIDATION / TEST SPLIT
# ---------------------------------------------------------

# First hold out 20% as the test set.
X_train_val, X_test, y_train_val, y_test, id_train_val, id_test = (
    train_test_split(
        X,
        y,
        subscriber_ids,
        test_size=0.20,
        stratify=y,
        random_state=42
    )
)

# Split the remaining 80% into 75% training and 25% validation.
X_train, X_val, y_train, y_val, id_train, id_val = (
    train_test_split(
        X_train_val,
        y_train_val,
        id_train_val,
        test_size=0.25,
        stratify=y_train_val,
        random_state=42
    )
)

print("\n========== DATA SPLIT ==========")
print("Training:", X_train.shape, "| churners:", int(y_train.sum()))
print("Validation:", X_val.shape, "| churners:", int(y_val.sum()))
print("Test:", X_test.shape, "| churners:", int(y_test.sum()))

print(
    "\nTraining is used to fit preprocessing and models; "
    "validation is used for model/threshold selection; "
    "test is used once for final evaluation."
)


# ---------------------------------------------------------
# 6. PREPROCESSING PIPELINE
# ---------------------------------------------------------

def make_preprocessor(numeric_columns, categorical_columns):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ])


preprocessor = make_preprocessor(
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES
)

# Fit only on training data.
X_train_transformed = preprocessor.fit_transform(X_train)
X_val_transformed = preprocessor.transform(X_val)

print("\n========== PREPROCESSING ==========")
print("Output feature count:", X_train_transformed.shape[1])
print("Missing values after transform:",
      np.isnan(X_train_transformed).sum())

numeric_imputer = (
    preprocessor.named_transformers_["numeric"]
    .named_steps["imputer"]
)

numeric_scaler = (
    preprocessor.named_transformers_["numeric"]
    .named_steps["scaler"]
)

age_index = NUMERIC_FEATURES.index("age")
tenure_index = NUMERIC_FEATURES.index("tenure_months")

print("Learned median age:", numeric_imputer.statistics_[age_index])
print("Scaler mean of tenure_months:", numeric_scaler.mean_[tenure_index])
print("Full-data mean of tenure_months:",
      subscribers["tenure_months"].mean())


# ---------------------------------------------------------
# 7. LEAKAGE EXPERIMENT
# ---------------------------------------------------------

print("\n========== DATA LEAKAGE EXPERIMENT ==========")

for col in LEAKAGE_FEATURES:
    if col not in subscribers.columns:
        raise ValueError(
            f"Expected leakage column '{col}' is missing from subscribers.csv."
        )

leakage_numeric = NUMERIC_FEATURES + ["next_cycle_recharge_inr"]
leakage_categorical = CATEGORICAL_FEATURES + ["closure_request_logged"]

X_leaky = subscribers[
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
    + LEAKAGE_FEATURES
].copy()

X_leak_train_val, X_leak_test, y_leak_train_val, y_leak_test = (
    train_test_split(
        X_leaky,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )
)

X_leak_train, X_leak_val, y_leak_train, y_leak_val = (
    train_test_split(
        X_leak_train_val,
        y_leak_train_val,
        test_size=0.25,
        stratify=y_leak_train_val,
        random_state=42
    )
)

leakage_preprocessor = make_preprocessor(
    leakage_numeric,
    leakage_categorical
)

leaky_pipeline = Pipeline([
    ("preprocessor", leakage_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

leaky_pipeline.fit(X_leak_train, y_leak_train)
leaky_predictions = leaky_pipeline.predict(X_leak_val)

normal_pipeline = Pipeline([
    ("preprocessor", make_preprocessor(
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES
    )),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

normal_pipeline.fit(X_train, y_train)
normal_predictions = normal_pipeline.predict(X_val)

print("WITH leakage:")
print("Accuracy:", round(accuracy_score(y_leak_val, leaky_predictions), 3))
print("Churn F1:", round(f1_score(y_leak_val, leaky_predictions), 3))

print("\nWITHOUT leakage:")
print("Accuracy:", round(accuracy_score(y_val, normal_predictions), 3))
print("Churn F1:", round(f1_score(y_val, normal_predictions), 3))

print(
    "\nLeakage columns are excluded from all final models because "
    "they are not available when a real prediction is made. "
    "Their use makes validation results unrealistically high and "
    "cannot represent performance on future subscribers."
)


# ---------------------------------------------------------
# 8. BASELINE AND SIMPLE MODELS
# ---------------------------------------------------------

print("\n========== BASELINE AND SIMPLE MODELS ==========")

# Majority-class baseline: always predicts that subscriber stays.
baseline_predictions = np.zeros(len(y_val), dtype=int)

print("\nMajority baseline:")
print("Accuracy:", round(accuracy_score(y_val, baseline_predictions), 3))
print("Churn F1:", round(f1_score(y_val, baseline_predictions), 3))

logistic_basic = Pipeline([
    ("preprocessor", make_preprocessor(
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES
    )),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_basic.fit(X_train, y_train)
logistic_basic_pred = logistic_basic.predict(X_val)

print("\nBasic Logistic Regression:")
print("Accuracy:", round(accuracy_score(y_val, logistic_basic_pred), 3))
print("Precision:", round(precision_score(y_val, logistic_basic_pred), 3))
print("Recall:", round(recall_score(y_val, logistic_basic_pred), 3))
print("F1:", round(f1_score(y_val, logistic_basic_pred), 3))

tree_basic = Pipeline([
    ("preprocessor", make_preprocessor(
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES
    )),
    ("model", DecisionTreeClassifier(random_state=42))
])

tree_basic.fit(X_train, y_train)
tree_train_pred = tree_basic.predict(X_train)
tree_val_pred = tree_basic.predict(X_val)

print("\nUnrestricted Decision Tree:")
print("Training accuracy:", round(accuracy_score(y_train, tree_train_pred), 3))
print("Validation F1:", round(f1_score(y_val, tree_val_pred), 3))

forest_basic = Pipeline([
    ("preprocessor", make_preprocessor(
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES
    )),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

forest_basic.fit(X_train, y_train)
forest_train_pred = forest_basic.predict(X_train)
forest_val_pred = forest_basic.predict(X_val)

print("\nDefault Random Forest:")
print("Training accuracy:", round(accuracy_score(y_train, forest_train_pred), 3))
print("Validation F1:", round(f1_score(y_val, forest_val_pred), 3))


# ---------------------------------------------------------
# 9. DECISION TREE DEPTH EXPERIMENT
# ---------------------------------------------------------

print("\n========== DECISION TREE DEPTH EXPERIMENT ==========")

depth_results = []

for depth in range(1, 11):
    depth_pipeline = Pipeline([
        ("preprocessor", make_preprocessor(
            NUMERIC_FEATURES,
            CATEGORICAL_FEATURES
        )),
        ("model", DecisionTreeClassifier(
            max_depth=depth,
            random_state=42
        ))
    ])

    depth_pipeline.fit(X_train, y_train)

    train_predictions = depth_pipeline.predict(X_train)
    validation_predictions = depth_pipeline.predict(X_val)

    depth_results.append({
        "max_depth": depth,
        "training_accuracy": accuracy_score(y_train, train_predictions),
        "validation_f1": f1_score(y_val, validation_predictions),
    })

depth_df = pd.DataFrame(depth_results)
depth_df.to_csv(OUTPUT_DIR / "decision_tree_depth.csv", index=False)

print(depth_df.round(3).to_string(index=False))

print(
    "\nAs tree depth increases, training accuracy can improve while "
    "validation F1 eventually declines. This widening gap indicates "
    "overfitting: a greedy tree can learn details and noise in its "
    "training data that do not generalise to unseen subscribers."
)


# ---------------------------------------------------------
# 10. GRID SEARCH: LOGISTIC REGRESSION
# ---------------------------------------------------------

print("\n========== LOGISTIC REGRESSION GRID SEARCH ==========")

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_pipeline = Pipeline([
    ("preprocessor", make_preprocessor(
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES
    )),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

logistic_search = GridSearchCV(
    logistic_pipeline,
    logistic_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

logistic_search.fit(X_train, y_train)

logistic_best = logistic_search.best_estimator_

logistic_best_index = logistic_search.best_index_
logistic_cv_mean = logistic_search.cv_results_["mean_test_score"][
    logistic_best_index
]
logistic_cv_std = logistic_search.cv_results_["std_test_score"][
    logistic_best_index
]

print("Best parameters:", logistic_search.best_params_)
print("Best mean CV F1:", round(logistic_cv_mean, 3))
print("CV F1 standard deviation:", round(logistic_cv_std, 3))
print("Combinations:", len(logistic_grid["model__C"]))
print("Model fits:", len(logistic_grid["model__C"]) * 5)


# ---------------------------------------------------------
# 11. GRID SEARCH: RANDOM FOREST
# ---------------------------------------------------------

print("\n========== RANDOM FOREST GRID SEARCH ==========")

forest_pipeline = Pipeline([
    ("preprocessor", make_preprocessor(
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES
    )),
    ("model", RandomForestClassifier(random_state=42))
])

forest_grid = {
    "model__n_estimators": [50, 100, 150],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__max_features": [3, 4, 5],
}

forest_search = GridSearchCV(
    forest_pipeline,
    forest_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

forest_search.fit(X_train, y_train)

forest_best = forest_search.best_estimator_

forest_best_index = forest_search.best_index_
forest_cv_mean = forest_search.cv_results_["mean_test_score"][
    forest_best_index
]
forest_cv_std = forest_search.cv_results_["std_test_score"][
    forest_best_index
]

forest_combinations = (
    len(forest_grid["model__n_estimators"])
    * len(forest_grid["model__max_depth"])
    * len(forest_grid["model__max_features"])
)

print("Best parameters:", forest_search.best_params_)
print("Best mean CV F1:", round(forest_cv_mean, 3))
print("CV F1 standard deviation:", round(forest_cv_std, 3))
print("Combinations:", forest_combinations)
print("Model fits:", forest_combinations * 5)

print(
    "\nThe model with the smaller cross-validation standard deviation "
    "is more consistent across folds. Consistency matters because its "
    "performance is less dependent on one particular training subset."
)


# ---------------------------------------------------------
# 12. THRESHOLD SELECTION USING VALIDATION COST
# ---------------------------------------------------------

print("\n========== VALIDATION THRESHOLD SELECTION ==========")

def calculate_cost(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    cost = (1200 * fn) + (300 * fp)

    return {
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "cost": cost,
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
    }


logistic_val_prob = logistic_best.predict_proba(X_val)[:, 1]
forest_val_prob = forest_best.predict_proba(X_val)[:, 1]

thresholds = np.round(np.arange(0.10, 0.901, 0.05), 2)

logistic_threshold_results = [
    calculate_cost(y_val, logistic_val_prob, t)
    for t in thresholds
]

forest_threshold_results = [
    calculate_cost(y_val, forest_val_prob, t)
    for t in thresholds
]

logistic_threshold_df = pd.DataFrame(logistic_threshold_results)
forest_threshold_df = pd.DataFrame(forest_threshold_results)

logistic_threshold_df.to_csv(
    OUTPUT_DIR / "logistic_thresholds.csv", index=False
)
forest_threshold_df.to_csv(
    OUTPUT_DIR / "forest_thresholds.csv", index=False
)

# Select the cheapest threshold from validation only.
best_logistic_threshold_row = (
    logistic_threshold_df.sort_values(
        ["cost", "threshold"]
    ).iloc[0]
)

best_forest_threshold_row = (
    forest_threshold_df.sort_values(
        ["cost", "threshold"]
    ).iloc[0]
)

logistic_threshold = float(
    best_logistic_threshold_row["threshold"]
)

forest_threshold = float(
    best_forest_threshold_row["threshold"]
)

print("Logistic best threshold:", logistic_threshold)
print("Logistic validation cost:",
      int(best_logistic_threshold_row["cost"]))

print("Random Forest best threshold:", forest_threshold)
print("Random Forest validation cost:",
      int(best_forest_threshold_row["cost"]))


# ---------------------------------------------------------
# 13. FINAL TEST EVALUATION
# ---------------------------------------------------------

print("\n========== FINAL TEST EVALUATION ==========")

def evaluate_test_model(model, threshold, X_test_data, y_test_data):
    probabilities = model.predict_proba(X_test_data)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_data,
        predictions,
        labels=[0, 1]
    ).ravel()

    total_cost = (1200 * fn) + (300 * fp)

    result = {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "precision": precision_score(
            y_test_data, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test_data, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_test_data, predictions, zero_division=0
        ),
        "cost": int(total_cost),
        "probabilities": probabilities,
        "predictions": predictions,
    }

    return result


logistic_test_result = evaluate_test_model(
    logistic_best,
    logistic_threshold,
    X_test,
    y_test
)

forest_test_result = evaluate_test_model(
    forest_best,
    forest_threshold,
    X_test,
    y_test
)

def print_test_result(name, result):
    print(f"\n{name}")
    print("TN / FP / FN / TP:",
          result["tn"], "/", result["fp"], "/",
          result["fn"], "/", result["tp"])
    print("Precision:", round(result["precision"], 3))
    print("Recall:", round(result["recall"], 3))
    print("F1:", round(result["f1"], 3))
    print("Total cost: INR", result["cost"])


print_test_result("LOGISTIC REGRESSION", logistic_test_result)
print_test_result("RANDOM FOREST", forest_test_result)

test_churners = int(y_test.sum())
no_action_cost = 1200 * test_churners
offer_everybody_cost = 300 * (len(y_test) - test_churners)

print("\nReference policy comparison:")
print("No action cost: INR", no_action_cost)
print("Offer everybody cost: INR", offer_everybody_cost)


# ---------------------------------------------------------
# 14. SELECT MODEL USING VALIDATION COST
# ---------------------------------------------------------

# Selection is based on validation cost, not test performance.
if best_forest_threshold_row["cost"] < best_logistic_threshold_row["cost"]:
    selected_name = "Random Forest"
    selected_model = forest_best
    selected_threshold = forest_threshold
    selected_test_result = forest_test_result
else:
    selected_name = "Logistic Regression"
    selected_model = logistic_best
    selected_threshold = logistic_threshold
    selected_test_result = logistic_test_result

print("\n========== SELECTED MODEL ==========")
print("Selected model:", selected_name)
print("Selected threshold:", selected_threshold)
print("Selected test cost: INR", selected_test_result["cost"])
print("Selected test recall:", round(selected_test_result["recall"], 3))
print("Selected test F1:", round(selected_test_result["f1"], 3))

print(
    "\nThe selected model is chosen using validation cost, not test "
    "performance. The test set is used only for the final evaluation "
    "after model and threshold selection."
)

if selected_test_result["cost"] > 74160:
    print("WARNING: Test cost is above the required INR 74,160.")
if selected_test_result["recall"] < 0.55:
    print("WARNING: Test recall is below the required 0.55.")
if selected_test_result["f1"] < 0.50:
    print("WARNING: Test F1 is below the required 0.50.")


# ---------------------------------------------------------
# 15. FEATURE IMPORTANCE AND LOGISTIC COEFFICIENTS
# ---------------------------------------------------------

print("\n========== FEATURE IMPORTANCE ==========")

forest_preprocessor = forest_best.named_steps["preprocessor"]
forest_model = forest_best.named_steps["model"]

feature_names = forest_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": forest_model.feature_importances_,
}).sort_values("importance", ascending=False)

print("\nTop five Random Forest features:")
print(importance_df.head(5).round(4).to_string(index=False))

importance_df.to_csv(
    OUTPUT_DIR / "random_forest_feature_importance.csv",
    index=False
)

print("\n========== LOGISTIC REGRESSION COEFFICIENTS ==========")

logistic_preprocessor = logistic_best.named_steps["preprocessor"]
logistic_model = logistic_best.named_steps["model"]

logistic_feature_names = logistic_preprocessor.get_feature_names_out()

coefficient_df = pd.DataFrame({
    "feature": logistic_feature_names,
    "coefficient": logistic_model.coef_[0],
})

requested_features = [
    "support_calls_90d",
    "tenure_months",
    "n_failed_6m",
]

for feature in requested_features:
    matching = coefficient_df[
        coefficient_df["feature"].str.endswith(feature)
    ]

    if not matching.empty:
        print(matching.to_string(index=False))

coefficient_df.to_csv(
    OUTPUT_DIR / "logistic_regression_coefficients.csv",
    index=False
)


# ---------------------------------------------------------
# 16. SAVE MODEL AND VERIFY RELOADING
# ---------------------------------------------------------

model_path = OUTPUT_DIR / "churn_model.joblib"

joblib.dump(
    {
        "model": selected_model,
        "threshold": selected_threshold,
        "model_name": selected_name,
    },
    model_path
)

loaded_package = joblib.load(model_path)
loaded_model = loaded_package["model"]
loaded_threshold = loaded_package["threshold"]

original_probabilities = selected_model.predict_proba(X_test)[:, 1]
loaded_probabilities = loaded_model.predict_proba(X_test)[:, 1]

original_predictions = (
    original_probabilities >= selected_threshold
).astype(int)

loaded_predictions = (
    loaded_probabilities >= loaded_threshold
).astype(int)

print("\nModel reload verification:")
print(
    "Predictions identical:",
    np.array_equal(original_predictions, loaded_predictions)
)


# ---------------------------------------------------------
# 17. SAVE TEST SCORES
# ---------------------------------------------------------

test_scores = pd.DataFrame({
    "subscriber_id": id_test.values,
    "churn_probability": original_probabilities,
    "churned": y_test.values,
})

test_scores = test_scores.sort_values("subscriber_id")

test_scores.to_csv(
    OUTPUT_DIR / "test_scores.csv",
    index=False
)

print("\nSaved:", model_path)
print("Saved:", OUTPUT_DIR / "test_scores.csv")
print("Test score rows:", len(test_scores))

print("\n========== PART B COMPLETED ==========")



========== PART B: CHURN PREDICTION ==========
Subscribers: (2400, 17)
Recharges: (12985, 6)

Recharge feature checks:
Missing recency_days: 76
Total successful recharges in 6 months: 11442
Total failed recharges in 6 months: 1422

========== CLASS BALANCE ==========
Stayed (0): 1886
Churned (1): 514
Churn percentage: 21.42 %

========== DATA SPLIT ==========
Training: (1440, 16) | churners: 308
Validation: (480, 16) | churners: 103
Test: (480, 16) | churners: 103

Training is used to fit preprocessing and models; validation is used for model/threshold selection; test is used once for final evaluation.

========== PREPROCESSING ==========
Output feature count: 31
Missing values after transform: 0
Learned median age: 38.0
Scaler mean of tenure_months: 22.65277777777778
Full-data mean of tenure_months: 22.46

========== DATA LEAKAGE EXPERIMENT ==========
WITH leakage:
Accuracy: 0.983
Churn F1: 0.961

WITHOUT leakage:
Accuracy: 0.823
Churn F1: 0.514

Leakage columns are excluded from all

In [ ]:

# ============================================================
# PART C: SEGMENTS, SPEND MODEL AND REVENUE AT RISK
# Tata Play Subscriber Retention & Support Copilot
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# ------------------------------------------------------------
# 1. PATHS AND INPUT FILES
# ------------------------------------------------------------

ROOT = Path("/content")
CLEAN_DIR = ROOT / "data" / "clean"
OUTPUT_DIR = ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBSCRIBER_FILE = CLEAN_DIR / "subscribers.csv"
TEST_SCORE_FILE = OUTPUT_DIR / "test_scores.csv"

if not SUBSCRIBER_FILE.exists():
    raise FileNotFoundError(f"Missing file: {SUBSCRIBER_FILE}")

if not TEST_SCORE_FILE.exists():
    raise FileNotFoundError(
        f"Missing file: {TEST_SCORE_FILE}. Run Part B successfully first."
    )

subscribers = pd.read_csv(SUBSCRIBER_FILE)
test_scores = pd.read_csv(TEST_SCORE_FILE)

print("Subscribers:", subscribers.shape)
print("Part B test scores:", test_scores.shape)

# Required columns
required_columns = [
    "subscriber_id",
    "tenure_months",
    "hours_watched_per_week",
    "monthly_recharge_inr",
    "churned",
    "next_cycle_recharge_inr",
    "hd_hours_per_week",
    "sports_hours_per_week",
    "channels_watched",
    "support_calls_90d"
]

missing_columns = [
    col for col in required_columns if col not in subscribers.columns
]

if missing_columns:
    raise ValueError(f"Missing subscriber columns: {missing_columns}")

# Convert relevant columns to numeric
for col in required_columns:
    if col != "subscriber_id":
        subscribers[col] = pd.to_numeric(
            subscribers[col], errors="coerce"
        )

# ------------------------------------------------------------
# 2. K-MEANS CLUSTERING
# ------------------------------------------------------------

print("\n========== K-MEANS CLUSTERING ==========")

cluster_features = [
    "tenure_months",
    "hours_watched_per_week",
    "monthly_recharge_inr"
]

cluster_data = subscribers[cluster_features].copy()

if cluster_data.isna().any().any():
    raise ValueError(
        "Missing values found in K-means features. "
        "Check the cleaned subscriber data before clustering."
    )

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data)

# Evaluate K from 2 to 8
kmeans_results = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )

    labels = model.fit_predict(X_scaled)

    inertia = model.inertia_
    silhouette = silhouette_score(X_scaled, labels)

    kmeans_results.append({
        "K": k,
        "Inertia": inertia,
        "Silhouette Score": silhouette
    })

kmeans_table = pd.DataFrame(kmeans_results)

print("\nK-means evaluation:")
print(kmeans_table.round(3).to_string(index=False))

# Choose K=4 using elbow, silhouette and business interpretability
BEST_K = 4

kmeans = KMeans(
    n_clusters=BEST_K,
    n_init=10,
    random_state=42
)

subscribers["segment_id"] = kmeans.fit_predict(X_scaled)

# ------------------------------------------------------------
# 3. PROFILE AND NAME THE SEGMENTS
# ------------------------------------------------------------

print("\n========== SEGMENT PROFILES ==========")

profile = subscribers.groupby("segment_id").agg(
    subscriber_count=("subscriber_id", "count"),
    mean_tenure=("tenure_months", "mean"),
    mean_weekly_hours=("hours_watched_per_week", "mean"),
    mean_monthly_recharge=("monthly_recharge_inr", "mean"),
    churn_rate=("churned", "mean")
).reset_index()

profile["churn_rate"] = profile["churn_rate"] * 100

print(profile.round(2).to_string(index=False))

# Name segments using their measured profiles.
# Churn is used only to describe/name clusters, not to build them.
at_risk_id = int(profile.loc[profile["churn_rate"].idxmax(), "segment_id"])

remaining = profile[profile["segment_id"] != at_risk_id]
premium_id = int(
    remaining.loc[
        remaining["mean_monthly_recharge"].idxmax(),
        "segment_id"
    ]
)

other_ids = [
    int(x) for x in profile["segment_id"]
    if int(x) not in [at_risk_id, premium_id]
]

# Give the remaining two clusters descriptive names by tenure.
other_profiles = (
    profile[profile["segment_id"].isin(other_ids)]
    .sort_values("mean_tenure")
)

segment_names = {
    at_risk_id: "At-Risk",
    premium_id: "Premium"
}

if len(other_profiles) >= 2:
    segment_names[int(other_profiles.iloc[0]["segment_id"])] = "New & Budget"
    segment_names[int(other_profiles.iloc[1]["segment_id"])] = "Established"

subscribers["segment_name"] = subscribers["segment_id"].map(segment_names)

profile["segment_name"] = profile["segment_id"].map(segment_names)

print("\nNamed segment profile:")
print(
    profile[
        [
            "segment_id",
            "segment_name",
            "subscriber_count",
            "mean_tenure",
            "mean_weekly_hours",
            "mean_monthly_recharge",
            "churn_rate"
        ]
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# 4. ASSIGN A NEW SUBSCRIBER BY HAND
# ------------------------------------------------------------

print("\n========== NEW SUBSCRIBER ASSIGNMENT ==========")

new_subscriber = pd.DataFrame([{
    "tenure_months": 4,
    "hours_watched_per_week": 15,
    "monthly_recharge_inr": 520
}])

new_scaled = scaler.transform(new_subscriber[cluster_features])

distances = np.linalg.norm(
    kmeans.cluster_centers_ - new_scaled,
    axis=1
)

nearest_cluster = int(np.argmin(distances))
predicted_cluster = int(kmeans.predict(new_scaled)[0])

print("Distances to centroids:")
for cluster_id, distance in enumerate(distances):
    print(
        f"Cluster {cluster_id} ({segment_names[cluster_id]}): "
        f"{distance:.4f}"
    )

print("Nearest cluster by hand:", nearest_cluster)
print("Nearest segment:", segment_names[nearest_cluster])
print("KMeans predict result:", predicted_cluster)

assert nearest_cluster == predicted_cluster, (
    "Manual nearest-centroid assignment does not match KMeans.predict."
)

# Save segments for all subscribers
segments_output = subscribers[
    ["subscriber_id", "segment_id", "segment_name"]
].copy()

segments_output.to_csv(
    OUTPUT_DIR / "segments.csv",
    index=False
)

print("\nSaved outputs/segments.csv")
print("Segment rows:", len(segments_output))

# ------------------------------------------------------------
# 5. CORRELATION MATRIX AND HIGH CORRELATION PAIRS
# ------------------------------------------------------------

print("\n========== CORRELATION ANALYSIS ==========")

correlation_features = [
    "hours_watched_per_week",
    "hd_hours_per_week",
    "sports_hours_per_week",
    "channels_watched"
]

correlation_matrix = subscribers[
    correlation_features
].corr()

print("\nCorrelation matrix:")
print(correlation_matrix.round(3))

print("\nPairs with absolute correlation above 0.9:")

for i in range(len(correlation_features)):
    for j in range(i + 1, len(correlation_features)):
        feature_a = correlation_features[i]
        feature_b = correlation_features[j]
        corr_value = correlation_matrix.loc[feature_a, feature_b]

        if abs(corr_value) > 0.9:
            print(
                f"{feature_a} - {feature_b}: {corr_value:.3f}"
            )

# ------------------------------------------------------------
# 6. PREPARE THE SPEND PREDICTION DATA
# ------------------------------------------------------------

print("\n========== SPEND PREDICTION ==========")

spend_features = [
    "monthly_recharge_inr",
    "tenure_months",
    "hours_watched_per_week",
    "hd_hours_per_week",
    "sports_hours_per_week",
    "channels_watched",
    "support_calls_90d"
]

# Only subscribers who did not churn are used for training/testing.
spend_data = subscribers[
    subscribers["churned"] == 0
].copy()

spend_data = spend_data.dropna(
    subset=spend_features + ["next_cycle_recharge_inr"]
)

X = spend_data[spend_features]
y = spend_data["next_cycle_recharge_inr"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Spend-model training rows:", len(X_train))
print("Spend-model testing rows:", len(X_test))

# ------------------------------------------------------------
# 7. LINEAR REGRESSION, RIDGE AND LASSO
# ------------------------------------------------------------

alpha_values = [
    0.001, 0.01, 0.1, 0.5, 1, 2, 3, 10, 30, 100
]

linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=10000))
])

# Linear Regression
linear_pipeline.fit(X_train, y_train)

# Ridge Grid Search
ridge_search = GridSearchCV(
    ridge_pipeline,
    {"model__alpha": alpha_values},
    cv=5,
    scoring="r2",
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)
best_ridge = ridge_search.best_estimator_

# Lasso Grid Search
lasso_search = GridSearchCV(
    lasso_pipeline,
    {"model__alpha": alpha_values},
    cv=5,
    scoring="r2",
    n_jobs=-1
)

lasso_search.fit(X_train, y_train)
best_lasso = lasso_search.best_estimator_

# ------------------------------------------------------------
# 8. MODEL EVALUATION
# ------------------------------------------------------------

def evaluate_regression(model, X_test, y_test):
    """Calculate R-squared, MAE and RMSE for a regression model."""
    predictions = model.predict(X_test)

    return {
        "Test R2": r2_score(y_test, predictions),
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions))
    }


linear_metrics = evaluate_regression(
    linear_pipeline, X_test, y_test
)

ridge_metrics = evaluate_regression(
    best_ridge, X_test, y_test
)

lasso_metrics = evaluate_regression(
    best_lasso, X_test, y_test
)

linear_cv = np.mean(
    # Cross-validation is run only on the training rows.
    __import__("sklearn.model_selection", fromlist=["cross_val_score"])
    .cross_val_score(
        linear_pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )
)

model_results = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "Best Alpha": "Not applicable",
        "CV R2": linear_cv,
        **linear_metrics
    },
    {
        "Model": "Ridge",
        "Best Alpha": ridge_search.best_params_["model__alpha"],
        "CV R2": ridge_search.best_score_,
        **ridge_metrics
    },
    {
        "Model": "Lasso",
        "Best Alpha": lasso_search.best_params_["model__alpha"],
        "CV R2": lasso_search.best_score_,
        **lasso_metrics
    }
])

print("\nRegression model comparison:")
print(model_results.round(4).to_string(index=False))

print("\nBest Ridge alpha:", ridge_search.best_params_["model__alpha"])
print("Best Lasso alpha:", lasso_search.best_params_["model__alpha"])

# Save best Ridge model
joblib.dump(best_ridge, OUTPUT_DIR / "spend_model.joblib")
print("\nSaved outputs/spend_model.joblib")

# ------------------------------------------------------------
# 9. LASSO ZERO COEFFICIENTS
# ------------------------------------------------------------

print("\n========== LASSO FEATURE SELECTION ==========")

lasso_coefficients = best_lasso.named_steps["model"].coef_

lasso_coefficient_table = pd.DataFrame({
    "Feature": spend_features,
    "Coefficient": lasso_coefficients
})

print(lasso_coefficient_table.to_string(index=False))

zero_features = lasso_coefficient_table.loc[
    lasso_coefficient_table["Coefficient"] == 0,
    "Feature"
].tolist()

print("\nFeatures with exactly zero Lasso coefficients:")
print(zero_features)

# ------------------------------------------------------------
# 10. STABILITY EXPERIMENT
# ------------------------------------------------------------

print("\n========== STABILITY EXPERIMENT ==========")

linear_hours_coefficients = []
ridge_hours_coefficients = []

for random_state in range(5):
    X_sample = X_train.sample(
        frac=0.8,
        random_state=random_state
    )

    y_sample = y_train.loc[X_sample.index]

    # Linear Regression
    linear_sample = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    linear_sample.fit(X_sample, y_sample)

    linear_hours_coefficients.append(
        linear_sample.named_steps["model"].coef_[
            spend_features.index("hours_watched_per_week")
        ]
    )

    # Ridge with alpha=30
    ridge_sample = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=30))
    ])

    ridge_sample.fit(X_sample, y_sample)

    ridge_hours_coefficients.append(
        ridge_sample.named_steps["model"].coef_[
            spend_features.index("hours_watched_per_week")
        ]
    )

linear_stability = np.std(
    linear_hours_coefficients,
    ddof=0
)

ridge_stability = np.std(
    ridge_hours_coefficients,
    ddof=0
)

print("Linear Regression coefficients:", linear_hours_coefficients)
print("Ridge coefficients:", ridge_hours_coefficients)
print("Linear Regression coefficient std:", round(linear_stability, 4))
print("Ridge coefficient std:", round(ridge_stability, 4))

# ------------------------------------------------------------
# 11. REVENUE AT RISK
# ------------------------------------------------------------

print("\n========== REVENUE AT RISK ==========")

# Attach segment information to Part B test scores.
test_data = test_scores.merge(
    subscribers[
        [
            "subscriber_id",
            "segment_id",
            "segment_name",
            "next_cycle_recharge_inr"
        ]
    ],
    on="subscriber_id",
    how="left",
    validate="one_to_one"
)

if test_data["segment_name"].isna().any():
    raise ValueError("Some Part B test subscribers have no segment assignment.")

# Predict next-cycle recharge for every Part B test subscriber,
# including subscribers whose actual outcome is churned.
X_priority = subscribers.set_index("subscriber_id").loc[
    test_data["subscriber_id"],
    spend_features
]

if X_priority.isna().any().any():
    raise ValueError(
        "Missing spend-model input values in Part B test subscribers."
    )

predicted_recharge = best_ridge.predict(X_priority)

test_data["predicted_next_recharge_inr"] = predicted_recharge

test_data["revenue_at_risk_inr"] = (
    test_data["churn_probability"]
    * test_data["predicted_next_recharge_inr"]
)

# Select top 15 subscribers by revenue at risk.
priority_list = test_data.sort_values(
    "revenue_at_risk_inr",
    ascending=False
).head(15).copy()

priority_columns = [
    "subscriber_id",
    "churn_probability",
    "predicted_next_recharge_inr",
    "revenue_at_risk_inr",
    "segment_name",
    "churned"
]

priority_list = priority_list[priority_columns]

priority_list.to_csv(
    OUTPUT_DIR / "retention_priority.csv",
    index=False
)

print("\nTop 15 revenue-at-risk subscribers:")
print(priority_list.round(2).to_string(index=False))

total_revenue_at_risk = priority_list[
    "revenue_at_risk_inr"
].sum()

actual_churners = int(priority_list["churned"].sum())

print("\nTotal revenue at risk for top 15: INR",
      round(total_revenue_at_risk, 2))

print("Number of actual churners in top 15:", actual_churners)

# ------------------------------------------------------------
# 12. FINAL OUTPUT CHECK
# ------------------------------------------------------------

print("\n========== PART C COMPLETED ==========")

print("Created files:")
for filename in [
    "segments.csv",
    "spend_model.joblib",
    "retention_priority.csv"
]:
    path = OUTPUT_DIR / filename
    print(filename, "->", "EXISTS" if path.exists() else "MISSING")

print("\nPart C completed successfully!")


Subscribers: (2400, 17)
Part B test scores: (480, 3)

========== K-MEANS CLUSTERING ==========

K-means evaluation:
 K  Inertia  Silhouette Score
 2 3728.219             0.420
 3 2812.958             0.346
 4 2193.933             0.340
 5 1781.092             0.343
 6 1502.476             0.344
 7 1292.830             0.364
 8 1191.417             0.336

========== SEGMENT PROFILES ==========
 segment_id  subscriber_count  mean_tenure  mean_weekly_hours  mean_monthly_recharge  churn_rate
          0               593        23.58               7.08                 301.33       22.43
          1               543        34.16              22.24                 559.73       13.44
          2               480        34.20              33.64                 942.32       10.62
          3               784         6.32              14.85                 513.51       32.78

Named segment profile:
 segment_id segment_name  subscriber_count  mean_tenure  mean_weekly_hours  mean_monthly_rechar

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving mock_llm.py to mock_llm (1).py


In [ ]:
import sys
sys.path.insert(0, "/content")

import mock_llm
print("mock_llm imported successfully!")

mock_llm imported successfully!


In [ ]:
!pip install -q "pydantic>=2,<3" langchain-core chromadb sentence-transformers

In [ ]:

%%writefile /content/llm_client.py

import os
import time
import mock_llm

BASE_DELAY = 0.05
REAL_URL = "https://openrouter.ai/api/v1/chat/completions"


def call_llm(messages, temperature=0.0, max_tokens=300):
    """Call the offline mock by default, or an optional configured live endpoint."""

    payload = {
        "model": "openai/gpt-4o-mini",
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    live_mode = os.getenv("LLM_MODE", "mock").lower() == "live"

    headers = {
        "Content-Type": "application/json"
    }

    if live_mode:
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise RuntimeError("OPENROUTER_API_KEY is not configured.")

        headers["Authorization"] = f"Bearer {api_key}"

        import requests
        post_function = requests.post
        url = REAL_URL
    else:
        headers["Authorization"] = "Bearer offline-mock-key"
        post_function = mock_llm.post
        url = mock_llm.URL

    for retry_number in range(4):
        response = post_function(
            url,
            headers=headers,
            json=payload,
            timeout=60
        )

        if response.status_code == 200:
            result = response.json()
            choice = result["choices"][0]

            if choice.get("finish_reason") == "length":
                print("Warning: response reached the token limit.")

            return choice["message"]["content"]

        if response.status_code == 429:
            if retry_number < 3:
                time.sleep(BASE_DELAY * (2 ** retry_number))
                continue

            raise RuntimeError("Rate limit exceeded after 3 retries.")

        if response.status_code == 400:
            raise RuntimeError(f"Bad request: {response.text}")

        if response.status_code == 401:
            raise RuntimeError(f"Authentication error: {response.text}")

        if response.status_code == 404:
            raise RuntimeError(f"Endpoint not found: {response.text}")

        raise RuntimeError(
            f"Unexpected API error {response.status_code}: {response.text}"
        )

    raise RuntimeError("Unable to complete LLM request.")


Overwriting /content/llm_client.py


In [ ]:

import sys
sys.path.insert(0, "/content")

import mock_llm
from llm_client import call_llm

# Test 1: Missing authorization
mock_llm.reset()

response = mock_llm.post(
    mock_llm.URL,
    headers={},
    json={
        "model": "openai/gpt-4o-mini",
        "messages": [{"role": "user", "content": "Hello"}]
    }
)

print("Missing authorization:", response.status_code)

# Test 2: Wrong URL
mock_llm.reset()

response = mock_llm.post(
    "https://wrong-url.example",
    headers={"Authorization": "Bearer test"},
    json={
        "model": "openai/gpt-4o-mini",
        "messages": [{"role": "user", "content": "Hello"}]
    }
)

print("Wrong URL:", response.status_code)

# Test 3: Token limit
mock_llm.reset()

reply = call_llm(
    [{"role": "user", "content": "Explain subscriber retention in detail."}],
    max_tokens=3
)

print("Short reply:", reply)

# Test 4: Memory demonstration
mock_llm.reset()

print(call_llm([
    {"role": "user", "content": "What is my name?"}
]))

mock_llm.reset()

print(call_llm([
    {"role": "user", "content": "My name is Asha."},
    {"role": "assistant", "content": "Hello Asha."},
    {"role": "user", "content": "What is my name?"}
]))

# Test 5: 30 calls and rate-limit recovery
mock_llm.reset()

for i in range(30):
    call_llm([
        {"role": "user", "content": f"Test request {i + 1}"}
    ])

print("Total mock calls:", mock_llm._state["calls"])


Missing authorization: 401
Wrong URL: 404
Short reply: I am
I do not know your name.
Your name is Asha
Total mock calls: 33


In [ ]:
import joblib
from pathlib import Path

model_path = Path("/content/outputs/churn_model.joblib")

assert model_path.exists(), "Run Part B first."

model = joblib.load(model_path)

print("Saved model type:", type(model))
print("Has predict_proba:", hasattr(model, "predict_proba"))
print("Has predict:", hasattr(model, "predict"))

Saved model type: <class 'dict'>
Has predict_proba: False
Has predict: False


In [ ]:
!pip install -q "pydantic>=2,<3" langchain-core chromadb sentence-transformers requests

In [ ]:
%%writefile /content/llm_client.py

import os
import time
import requests
import mock_llm

API_URL = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(messages, temperature=0.0, max_tokens=300, retries=4):
    payload = {
        "model": "mock-model",
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    mode = os.getenv("LLM_MODE", "mock").lower()

    if mode == "live":
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise ValueError("Set OPENROUTER_API_KEY before using live mode.")

        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }

        def send():
            return requests.post(
                API_URL, headers=headers, json=payload, timeout=60
            )
    else:
        headers = {
            "Authorization": "Bearer local-mock-key",
            "Content-Type": "application/json"
        }

        def send():
            return mock_llm.post(
                mock_llm.URL, headers=headers, json=payload, timeout=60
            )

    for attempt in range(retries):
        response = send()

        if response.status_code == 200:
            body = response.json()
            return body["choices"][0]["message"]["content"]

        if response.status_code == 429 and attempt < retries - 1:
            time.sleep(0.1 * (attempt + 1))
            continue

        raise RuntimeError(
            f"LLM request failed: HTTP {response.status_code} - "
            f"{response.text}"
        )

    raise RuntimeError("LLM request failed after retries.")

Overwriting /content/llm_client.py


In [ ]:
import sys
sys.path.insert(0, "/content")

import mock_llm
from llm_client import call_llm

mock_llm.reset()

test_messages = [
    {
        "role": "system",
        "content": "You are a simple assistant."
    },
    {
        "role": "user",
        "content": "Hello"
    }
]

print(call_llm(test_messages))

I am a mock assistant. I can classify tickets, extract ticket fields, answer policy questions and write retention briefs.


In [ ]:
%%writefile /content/part_d.py

import os
import re
import json
import glob
import sys
import pandas as pd
import chromadb

from typing import Optional, Literal
from pydantic import BaseModel, Field, ValidationError
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

sys.path.insert(0, "/content")
from llm_client import call_llm

ROOT = "/content"
DATA = f"{ROOT}/data"
OUTPUT = f"{ROOT}/outputs"
POLICY_DIR = f"{DATA}/policies"

os.makedirs(OUTPUT, exist_ok=True)


# ---------------------------------------------------------
# GENERAL HELPERS
# ---------------------------------------------------------

def read_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}")
    return pd.read_csv(path)


def find_column(df, possible_names, required=True):
    lookup = {str(c).strip().lower(): c for c in df.columns}

    for name in possible_names:
        if name.lower() in lookup:
            return lookup[name.lower()]

    if required:
        raise ValueError(
            f"Could not find any of {possible_names}. "
            f"Available columns: {list(df.columns)}"
        )
    return None


def response_text(messages, max_tokens=300):
    return call_llm(messages, temperature=0, max_tokens=max_tokens)


# ---------------------------------------------------------
# 1. LOAD TICKETS AND FEW-SHOT EXAMPLES
# ---------------------------------------------------------

tickets = read_csv(f"{DATA}/raw/tickets.csv")
examples = read_csv(f"{DATA}/raw/few_shot_examples.csv")

ticket_text_col = find_column(tickets, ["text", "ticket_text", "message"])
ticket_id_col = find_column(tickets, ["ticket_id"], required=False)
subscriber_col = find_column(tickets, ["subscriber_id"], required=False)
true_label_col = find_column(
    tickets, ["true_category", "category", "label"], required=False
)

example_text_col = find_column(
    examples, ["text", "ticket_text", "ticket", "message"]
)
example_label_col = find_column(
    examples, ["label", "category", "true_category"]
)

few_shot_text = "\n\n".join(
    f"Ticket: {row[example_text_col]}\nLabel: {row[example_label_col]}"
    for _, row in examples.iterrows()
)


# ---------------------------------------------------------
# 2. TICKET CLASSIFICATION
# ---------------------------------------------------------

def classify_ticket(ticket, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": str(ticket)}
    ]
    return response_text(messages, max_tokens=100).strip()


zero_shot_system = """
TICKET CLASSIFIER.
You classify Tata Play customer support tickets.
Valid labels are: billing, signal, plan_change, cancellation.
Respond with only the label. Do not explain.
"""

few_shot_system = f"""
TICKET CLASSIFIER.
You classify Tata Play customer support tickets.
Valid labels are: billing, signal, plan_change, cancellation.

Use these labelled examples:
{few_shot_text}

Respond with only the label. Do not explain.
"""

weak_system = """
TICKET CLASSIFIER.
Classify the customer's support ticket.
"""


classification_rows = []

for index, row in tickets.iterrows():
    ticket = str(row[ticket_text_col]).strip()

    ticket_id = (
        row[ticket_id_col]
        if ticket_id_col
        else f"TICKET_{index + 1:03d}"
    )

    subscriber_id = (
        row[subscriber_col]
        if subscriber_col
        else None
    )

    true_category = (
        row[true_label_col]
        if true_label_col
        else None
    )

    if ticket in ("", "...", "nan", "None"):
        classification_rows.append({
            "ticket_id": ticket_id,
            "subscriber_id": subscriber_id,
            "ticket_text": ticket,
            "true_category": true_category,
            "zero_shot_prediction": None,
            "few_shot_prediction": None,
            "weak_prompt_response": None,
            "manual_review": True
        })
        continue

    zero = classify_ticket(ticket, zero_shot_system)
    few = classify_ticket(ticket, few_shot_system)
    weak = classify_ticket(ticket, weak_system)

    valid_labels = {
        "billing", "signal", "plan_change", "cancellation"
    }

    zero = zero.lower().strip()
    few = few.lower().strip()

    classification_rows.append({
        "ticket_id": ticket_id,
        "subscriber_id": subscriber_id,
        "ticket_text": ticket,
        "true_category": true_category,
        "zero_shot_prediction": zero if zero in valid_labels else None,
        "few_shot_prediction": few if few in valid_labels else None,
        "weak_prompt_response": weak,
        "manual_review": False
    })

classification_df = pd.DataFrame(classification_rows)
classification_df.to_csv(
    f"{OUTPUT}/ticket_classification.csv", index=False
)

print("Ticket classification completed.")
print(classification_df.head())


# ---------------------------------------------------------
# 3. STRUCTURED TICKET EXTRACTION WITH PYDANTIC
# ---------------------------------------------------------

class TicketRecord(BaseModel):
    subscriber_id: Optional[str] = None
    category: Literal[
        "billing", "signal", "plan_change", "cancellation"
    ]
    urgency: int = Field(ge=1, le=5)
    wants_refund: bool


extractor_system = """
TICKET EXTRACTOR.
Extract the ticket information and return ONLY a valid JSON object.

Required fields:
subscriber_id: subscriber ID such as TP00001, or null
category: billing, signal, plan_change, or cancellation
urgency: integer from 1 to 5
wants_refund: true or false

Do not add explanations or Markdown.
"""


def extract_ticket(ticket, max_retries=3):
    messages = [
        {"role": "system", "content": extractor_system},
        {"role": "user", "content": str(ticket)}
    ]

    last_error = None

    for attempt in range(max_retries):
        raw = response_text(messages, max_tokens=200)

        try:
            # Validate JSON structure and field constraints.
            record = TicketRecord.model_validate_json(raw)
            return record.model_dump(), raw, attempt + 1

        except Exception as exc:
            last_error = str(exc)

            messages.append({
                "role": "user",
                "content": (
                    "VALIDATION ERROR: "
                    + last_error
                    + ". Return a corrected JSON object only. "
                    "Use valid category values and urgency from 1 to 5."
                )
            })

    return None, str(last_error), max_retries


extraction_rows = []

for index, row in tickets.iterrows():
    ticket = str(row[ticket_text_col]).strip()

    ticket_id = (
        row[ticket_id_col]
        if ticket_id_col
        else f"TICKET_{index + 1:03d}"
    )

    if ticket in ("", "...", "nan", "None"):
        extraction_rows.append({
            "ticket_id": ticket_id,
            "ticket_text": ticket,
            "status": "manual_review",
            "attempts": 0,
            "extraction": None,
            "error": "Unreadable or empty ticket"
        })
        continue

    record, raw, attempts = extract_ticket(ticket)

    extraction_rows.append({
        "ticket_id": ticket_id,
        "ticket_text": ticket,
        "status": "success" if record else "manual_review",
        "attempts": attempts,
        "extraction": record,
        "error": None if record else raw
    })


with open(f"{OUTPUT}/ticket_extractions.jsonl", "w", encoding="utf-8") as file:
    for item in extraction_rows:
        file.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Ticket extraction completed.")
print("Successful extractions:", sum(
    x["status"] == "success" for x in extraction_rows
))
print("Manual review:", sum(
    x["status"] == "manual_review" for x in extraction_rows
))


# ---------------------------------------------------------
# 4. LOAD POLICY DOCUMENTS
# ---------------------------------------------------------

policy_paths = sorted(glob.glob(f"{POLICY_DIR}/*.txt"))

if not policy_paths:
    raise FileNotFoundError(
        f"No policy text files found in {POLICY_DIR}"
    )

policy_documents = []

for path in policy_paths:
    with open(path, "r", encoding="utf-8") as file:
        content = file.read()

    policy_documents.append({
        "filename": os.path.basename(path),
        "content": content
    })


# ---------------------------------------------------------
# 5. CREATE CHROMA VECTOR DATABASE
# ---------------------------------------------------------

embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("tataplay_policy_collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="tataplay_policy_collection",
    embedding_function=embedding_function
)

collection.add(
    ids=[f"policy_{i}" for i in range(len(policy_documents))],
    documents=[
        f"[{doc['filename']}]\n{doc['content']}"
        for doc in policy_documents
    ],
    metadatas=[
        {"filename": doc["filename"]}
        for doc in policy_documents
    ]
)


def retrieve_policy_context(question, top_k=5):
    count = min(top_k, len(policy_documents))

    result = collection.query(
        query_texts=[str(question)],
        n_results=count
    )

    docs = result["documents"][0]
    metadata = result["metadatas"][0]

    context = "\n\n".join(docs)
    filenames = [m["filename"] for m in metadata]

    return context, filenames


# ---------------------------------------------------------
# 6. POLICY QUESTION ANSWERING / RAG EVALUATION
# ---------------------------------------------------------

policy_eval = read_csv(f"{DATA}/raw/policy_eval.csv")

question_col = find_column(
    policy_eval, ["question", "query", "user_question"]
)

expected_file_col = find_column(
    policy_eval,
    ["expected_policy_file", "policy_file", "expected_file", "source_file"],
    required=False
)

expected_fact_col = find_column(
    policy_eval,
    ["expected_fact", "expected_answer", "expected_text", "answer"],
    required=False
)

rag_rows = []

policy_system = """
POLICY ASSISTANT.
Answer using only the retrieved context.
Do not use outside knowledge or invent policy details.
If the answer is not available in the context, clearly say:
I don't have enough information to answer that.
"""


for _, row in policy_eval.iterrows():
    question = str(row[question_col])

    expected_file = (
        row[expected_file_col]
        if expected_file_col else None
    )

    expected_fact = (
        row[expected_fact_col]
        if expected_fact_col else None
    )

    context, retrieved_files = retrieve_policy_context(question)

    user_message = (
        "CONTEXT:\n"
        + context
        + "\n\nQUESTION:\n"
        + question
    )

    answer = response_text([
        {"role": "system", "content": policy_system},
        {"role": "user", "content": user_message}
    ], max_tokens=250)

    expected_fact_text = (
        str(expected_fact)
        if pd.notna(expected_fact) and expected_fact is not None
        else ""
    )

    contains_fact = (
        expected_fact_text.strip().lower() in answer.lower()
        if expected_fact_text.strip()
        else None
    )

    rag_rows.append({
        "question": question,
        "expected_policy_file": expected_file,
        "retrieved_policy_files": "; ".join(retrieved_files),
        "answer": answer,
        "expected_fact": expected_fact,
        "answer_contains_expected_fact": contains_fact,
        "abstained": "don't have enough information" in answer.lower()
    })


rag_df = pd.DataFrame(rag_rows)
rag_df.to_csv(f"{OUTPUT}/rag_eval.csv", index=False)

print("RAG evaluation completed.")
print(rag_df.head())

Overwriting /content/part_d.py


In [ ]:
import sys
sys.path.insert(0, "/content")

import mock_llm
mock_llm.reset()

%run /content/part_d.py
%run /content/copilot.py

print("\nPART D EXECUTION FINISHED")

Ticket classification completed.
  ticket_id subscriber_id                                        ticket_text  \
0     T0001       TP02169  Subscriber ID TP02169. Please cancel my subscr...   
1     T0002       TP01419  Subscriber ID TP01419. The picture is fine but...   
2     T0003       TP01581  I am closing the account because the price is ...   
3     T0004       TP01501  Subscriber ID TP01501. There is no signal on m...   
4     T0005       TP02291  Auto renew took money from my account but the ...   

  true_category zero_shot_prediction few_shot_prediction  \
0  cancellation         cancellation        cancellation   
1       billing              billing             billing   
2  cancellation         cancellation        cancellation   
3        signal               signal              signal   
4       billing              billing             billing   

                                weak_prompt_response  manual_review  
0  This looks like a cancellation ticket because ...   

In [ ]:
from pathlib import Path
import pandas as pd
import json

output_dir = Path("/content/outputs")

required_files = [
    "ticket_classification.csv",
    "ticket_extractions.jsonl",
    "rag_eval.csv",
    "retention_briefs.jsonl"
]

for filename in required_files:
    path = output_dir / filename
    print(
        filename,
        "->",
        "CREATED" if path.exists() else "MISSING"
    )

print("\nClassification rows:",
      len(pd.read_csv(output_dir / "ticket_classification.csv")))

print("RAG evaluation rows:",
      len(pd.read_csv(output_dir / "rag_eval.csv")))

with open(output_dir / "ticket_extractions.jsonl", encoding="utf-8") as f:
    print("Ticket extraction records:", sum(1 for _ in f))

with open(output_dir / "retention_briefs.jsonl", encoding="utf-8") as f:
    print("Retention briefs:", sum(1 for _ in f))

ticket_classification.csv -> CREATED
ticket_extractions.jsonl -> CREATED
rag_eval.csv -> CREATED
retention_briefs.jsonl -> CREATED

Classification rows: 150
RAG evaluation rows: 16
Ticket extraction records: 150
Retention briefs: 15


In [ ]:
%%writefile /content/copilot.py

import os
import sys
import glob
import json
import pandas as pd

sys.path.insert(0, "/content")
from llm_client import call_llm

ROOT = "/content"
DATA = f"{ROOT}/data"
OUTPUT = f"{ROOT}/outputs"

os.makedirs(OUTPUT, exist_ok=True)


def find_column(df, names):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    raise ValueError(
        f"Missing column {names}. Available: {list(df.columns)}"
    )


# Load existing project outputs
subscribers = pd.read_csv(f"{DATA}/clean/subscribers.csv")
priority = pd.read_csv(f"{OUTPUT}/retention_priority.csv")
recharges = pd.read_csv(f"{DATA}/clean/recharges.csv")

sid_sub = find_column(subscribers, ["subscriber_id"])
sid_priority = find_column(priority, ["subscriber_id"])
sid_recharge = find_column(recharges, ["subscriber_id"])

tenure_col = find_column(subscribers, ["tenure_months"])
prob_col = find_column(
    priority,
    ["churn_probability", "predicted_churn_probability", "probability"]
)

status_col = find_column(recharges, ["status", "recharge_status"])

# Count failed recharges
failed = recharges[
    recharges[status_col].astype(str).str.lower().isin(
        ["failed", "failure"]
    )
]

failed_counts = failed.groupby(sid_recharge).size().to_dict()

# Select top 15 subscribers
priority = priority.sort_values(prob_col, ascending=False)
top15 = priority.head(15)

subscriber_lookup = subscribers.set_index(sid_sub).to_dict("index")

# Load policy files
policy_paths = sorted(glob.glob(f"{DATA}/policies/*.txt"))

if not policy_paths:
    raise FileNotFoundError("Policy files not found.")

policy_context = ""

for path in policy_paths:
    with open(path, "r", encoding="utf-8") as f:
        policy_context += (
            f"\n[{os.path.basename(path)}]\n{f.read()}\n"
        )

system_prompt = """
RETENTION BRIEF.
Use only the supplied subscriber facts and POLICY CONTEXT.
Return only a valid JSON object with:
subscriber_id, risk_band, recommended_action, policy_source.
Do not invent policy details.
"""

briefs = []

for _, row in top15.iterrows():

    sid = str(row[sid_priority])
    subscriber = subscriber_lookup.get(sid, {})

    tenure = int(float(subscriber.get(tenure_col, 0)))
    failed_count = int(failed_counts.get(sid, 0))

    probability = float(row[prob_col])

    risk_band = "High" if probability >= 0.5 else "Low"

    user_message = (
        f"subscriber_id: {sid}\n"
        f"risk_band: {risk_band}\n"
        f"tenure_months: {tenure}\n"
        f"failed_recharges_6m: {failed_count}\n"
        f"POLICY CONTEXT:\n{policy_context}"
    )

    answer = call_llm(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0,
        max_tokens=250
    )

    try:
        result = json.loads(answer)
    except json.JSONDecodeError:
        result = {
            "subscriber_id": sid,
            "risk_band": risk_band,
            "recommended_action": "Manual review required.",
            "policy_source": "none",
            "raw_response": answer
        }

    result["tenure_months"] = tenure
    result["failed_recharges_6m"] = failed_count
    result["churn_probability"] = probability

    briefs.append(result)


with open(
    f"{OUTPUT}/retention_briefs.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for brief in briefs:
        f.write(json.dumps(brief, ensure_ascii=False) + "\n")

print("Retention briefs generated successfully!")
print("Total subscribers:", len(briefs))
print(json.dumps(briefs[0], indent=2) if briefs else "No records generated.")

Overwriting /content/copilot.py


In [ ]:
%run /content/copilot.py

Retention briefs generated successfully!
Total subscribers: 15
{
  "subscriber_id": "TP00798",
  "risk_band": "High",
  "recommended_action": "Offer an auto-renew setup call and a 7 day grace extension.",
  "policy_source": "retention_offers_policy.txt",
  "tenure_months": 13,
  "failed_recharges_6m": 3,
  "churn_probability": 0.8176170892177579
}


In [ ]:
import os

files = [
    "ticket_classification.csv",
    "ticket_extractions.jsonl",
    "rag_eval.csv",
    "retention_briefs.jsonl"
]

for name in files:
    path = f"/content/outputs/{name}"
    print(name, ":", "SUCCESS" if os.path.exists(path) else "MISSING")

ticket_classification.csv : SUCCESS
ticket_extractions.jsonl : SUCCESS
rag_eval.csv : SUCCESS
retention_briefs.jsonl : SUCCESS


In [ ]:
import os

required_files = [
    "data/clean/subscribers.csv",
    "data/clean/recharges.csv",
    "data/clean/plans.csv",
    "data/tataplay.db",
    "sql/queries.sql",
    "outputs/churn_model.joblib",
    "outputs/test_scores.csv",
    "outputs/segments.csv",
    "outputs/retention_priority.csv",
    "outputs/ticket_classification.csv",
    "outputs/ticket_extractions.jsonl",
    "outputs/rag_eval.csv",
    "outputs/retention_briefs.jsonl"
]

for file in required_files:
    path = "/content/" + file
    print(("✅ " if os.path.exists(path) else "❌ ") + file)

✅ data/clean/subscribers.csv
✅ data/clean/recharges.csv
✅ data/clean/plans.csv
✅ data/tataplay.db
✅ sql/queries.sql
✅ outputs/churn_model.joblib
✅ outputs/test_scores.csv
✅ outputs/segments.csv
✅ outputs/retention_priority.csv
✅ outputs/ticket_classification.csv
✅ outputs/ticket_extractions.jsonl
✅ outputs/rag_eval.csv
✅ outputs/retention_briefs.jsonl


In [ ]:
import os

print("PROJECT FILES")
print("=" * 50)

for folder in ["data", "outputs", "sql"]:
    print(f"\n📁 {folder}/")
    if os.path.exists(folder):
        for root, dirs, files in os.walk(folder):
            dirs[:] = [d for d in dirs if d not in ["__pycache__", ".ipynb_checkpoints"]]
            for file in files:
                path = os.path.join(root, file)
                print("   ", path)
    else:
        print("   Folder not found")

print("\nROOT FILES")
print("=" * 50)

for file in os.listdir("."):
    if os.path.isfile(file):
        print(file)

PROJECT FILES

📁 data/
    data/tataplay.db
    data/policies/retention_offers_policy.txt
    data/policies/cancellation_policy.txt
    data/policies/technical_support_policy.txt
    data/policies/plan_change_policy.txt
    data/policies/recharge_and_refund_policy.txt
    data/raw/policy_eval.csv
    data/raw/plans.csv
    data/raw/recharges.csv
    data/raw/tickets.csv
    data/raw/subscribers.csv
    data/raw/few_shot_examples.csv
    data/clean/plans.csv
    data/clean/recharges.csv
    data/clean/subscribers.csv

📁 outputs/
    outputs/q3_ranking_within_city.csv
    outputs/retention_priority.csv
    outputs/q4_running_total.csv
    outputs/random_forest_feature_importance.csv
    outputs/ticket_extractions.jsonl
    outputs/retention_briefs.jsonl
    outputs/logistic_thresholds.csv
    outputs/logistic_regression_coefficients.csv
    outputs/q1_revenue_by_plan.csv
    outputs/forest_thresholds.csv
    outputs/spend_model.joblib
    outputs/decision_tree_depth.csv
    outputs/ticke

In [45]:
import os

print("Checking project files...")

for folder in ["data", "outputs", "sql"]:
    print(f"\n{folder}:")
    for root, dirs, files in os.walk(folder):
        for file in files:
            print(os.path.join(root, file))

print("\nProject file check completed!")

Checking project files...

data:
data/tataplay.db
data/policies/retention_offers_policy.txt
data/policies/cancellation_policy.txt
data/policies/technical_support_policy.txt
data/policies/plan_change_policy.txt
data/policies/recharge_and_refund_policy.txt
data/raw/policy_eval.csv
data/raw/plans.csv
data/raw/recharges.csv
data/raw/tickets.csv
data/raw/subscribers.csv
data/raw/few_shot_examples.csv
data/clean/plans.csv
data/clean/recharges.csv
data/clean/subscribers.csv

outputs:
outputs/q3_ranking_within_city.csv
outputs/retention_priority.csv
outputs/q4_running_total.csv
outputs/random_forest_feature_importance.csv
outputs/ticket_extractions.jsonl
outputs/retention_briefs.jsonl
outputs/logistic_thresholds.csv
outputs/logistic_regression_coefficients.csv
outputs/q1_revenue_by_plan.csv
outputs/forest_thresholds.csv
outputs/spend_model.joblib
outputs/decision_tree_depth.csv
outputs/ticket_classification.csv
outputs/test_scores.csv
outputs/q2_never_successfully_recharged.csv
outputs/rag_eva

In [46]:
import os
import zipfile
from google.colab import files

zip_name = "/content/Tata_Play_Retention_Copilot.zip"

include_folders = ["data", "outputs", "sql"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    # Add project folders
    for folder in include_folders:
        if os.path.exists(folder):
            for root, dirs, filenames in os.walk(folder):
                dirs[:] = [
                    d for d in dirs
                    if d not in ["__pycache__", ".ipynb_checkpoints"]
                ]
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, path)

    # Add Python files
    for filename in os.listdir("/content"):
        if filename.endswith(".py"):
            z.write(filename, filename)

print("Project ZIP created successfully!")
print("File:", zip_name)

# Download ZIP to your computer
files.download(zip_name)

Project ZIP created successfully!
File: /content/Tata_Play_Retention_Copilot.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:

from google.colab import drive
import os
import json
from pathlib import Path

# Connect Google Drive
drive.mount('/content/drive')

# Find your saved notebook
matches = []

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    dirs[:] = [d for d in dirs if d not in ['.ipynb_checkpoints']]
    if 'Untitled16.ipynb' in files:
        matches.append(os.path.join(root, 'Untitled16.ipynb'))

print("Notebook files found:", matches)

if len(matches) != 1:
    print("Please share the result with me before continuing.")
else:
    notebook_path = matches[0]

    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = json.load(f)

    # Extract the actual Part A, B and C code cells
    for cell_number, filename in [
        (0, 'part_a.py'),
        (1, 'part_b.py'),
        (2, 'part_c.py')
    ]:
        source = ''.join(notebook['cells'][cell_number]['source'])

        # Make the scripts work from the project folder
        source = source.replace(
            'ROOT = Path("/content")',
            'ROOT = Path(__file__).resolve().parent'
        )

        with open('/content/' + filename, 'w', encoding='utf-8') as f:
            f.write(source)

        print(filename, "created successfully!")

    print("\nPart A, B and C Python files are ready!")


Mounted at /content/drive
Notebook files found: ['/content/drive/MyDrive/Colab Notebooks/Untitled16.ipynb']
part_a.py created successfully!
part_b.py created successfully!
part_c.py created successfully!

Part A, B and C Python files are ready!


In [48]:
import os

for f in ["part_a.py", "part_b.py", "part_c.py"]:
    print(f, "✅ Found" if os.path.exists(f) else "❌ Missing")

part_a.py ✅ Found
part_b.py ✅ Found
part_c.py ✅ Found


In [49]:
requirements = """pandas
numpy
scikit-learn
joblib
pydantic>=2,<3
langchain-core
chromadb
sentence-transformers
requests
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [50]:
gitignore = """__pycache__/
*.pyc
.ipynb_checkpoints/
.env
.venv/
venv/
"""

with open(".gitignore", "w") as f:
    f.write(gitignore)

print(".gitignore created successfully!")

.gitignore created successfully!


In [51]:
readme = """# Tata Play Subscriber Retention & Support Copilot

## Overview

This project develops a data-driven subscriber retention and customer support copilot using synthetic Tata Play scenario data.

It combines data cleaning, SQL analysis, machine learning, customer segmentation, ticket classification, information extraction, retrieval-augmented generation (RAG), and retention recommendations.

All data used in this project is synthetic. It does not contain actual Tata Play customer records or represent official Tata Play policies.

## Reproduce

Requirements:
- Python 3.11 or 3.12
- SQLite 3.25 or later

Install dependencies:

    pip install -r requirements.txt

Run the project scripts in order:

    python generate_data.py
    python part_a.py
    python part_b.py
    python part_c.py
    python part_d.py
    python copilot.py

The scripts generate cleaned datasets, a SQLite database, model files, analysis outputs, ticket classifications, RAG evaluation results, and retention briefs.

## Part A

### Data Cleaning and SQL Analysis

- Generated synthetic subscriber, recharge, ticket, plan, and policy data.
- Raw subscriber records: 2,415.
- Duplicate subscriber records: 15.
- Clean subscriber records: 2,400.
- Raw recharge records: 13,055.
- Duplicate recharge records: 60.
- Clean recharge records: 12,985.
- Missing age values: 104.
- Missing payment mode values: 114.
- Malformed recharge amounts rebuilt: 364.
- Churned subscribers: 514 (21.42%).
- Successful recharge revenue: INR 6,457,573.
- Subscribers with no successful recharge: 76.

SQL analysis includes revenue by plan, subscribers without successful recharges, city-wise ranking, and running recharge totals.

## Part B

### Churn Prediction

Machine learning models were developed to predict subscriber churn.

- Target variable: churned.
- Total subscribers: 2,400.
- Training records: 1,440.
- Validation records: 480.
- Test records: 480.
- Baseline model accuracy: 78.5%.
- Logistic regression validation accuracy: 83.3%.
- Logistic regression validation precision: 64.6%.
- Logistic regression validation recall: 49.5%.
- Logistic regression validation F1-score: 56.0%.

The workflow includes preprocessing, feature engineering, model evaluation, and threshold selection using validation data.

## Part C

### Customer Segmentation and Revenue Analysis

- Applied K-Means clustering to create four customer segments.
- Developed spend prediction models using Linear Regression, Ridge, and Lasso.
- Estimated revenue at risk using churn probability and predicted next recharge.
- Generated subscriber retention priorities.

Generated outputs include:
- segments.csv
- spend_model.joblib
- retention_priority.csv

## Part D

### Support Copilot and RAG

Implemented ticket classification, structured information extraction, policy retrieval, and retention brief generation.

- Ticket records processed: 150.
- Successfully extracted ticket records: 146.
- Records requiring manual review: 4.
- RAG evaluation questions: 16.
- Retention briefs generated: 15.

The system uses a mock language model by default and supports policy-based retrieval and structured output validation.

## Limitations and Next Steps

- The dataset is synthetic and does not represent real customer behaviour.
- Model performance may differ on real-world data.
- The mock language model is intended for demonstration and testing.
- Retention recommendations should be reviewed by a human before action.
- Future improvements include testing with real, authorized data, improving model evaluation, refining policy retrieval, and adding monitoring.

"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README.md created successfully!")

README.md created successfully!


In [52]:
import os

files = [
    "generate_data.py",
    "mock_llm.py",
    "part_a.py",
    "part_b.py",
    "part_c.py",
    "part_d.py",
    "llm_client.py",
    "copilot.py",
    "requirements.txt",
    ".gitignore",
    "README.md",
    "sql/queries.sql"
]

for file in files:
    print(("✅ " if os.path.exists(file) else "❌ ") + file)

✅ generate_data.py
✅ mock_llm.py
✅ part_a.py
✅ part_b.py
✅ part_c.py
✅ part_d.py
✅ llm_client.py
✅ copilot.py
✅ requirements.txt
✅ .gitignore
✅ README.md
✅ sql/queries.sql


In [53]:
import os
import zipfile

zip_name = "Tata_Play_Retention_Copilot_Final.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

        for file in files:
            if file.endswith(".ipynb") or file == zip_name:
                continue

            path = os.path.join(root, file)
            z.write(path, os.path.relpath(path, "."))

print("Final project ZIP created:", zip_name)

Final project ZIP created: Tata_Play_Retention_Copilot_Final.zip


In [54]:
import glob
import os
import nbformat

notebooks = glob.glob("/content/*.ipynb")

print("Notebooks found:")
for i, path in enumerate(notebooks):
    print(i, os.path.basename(path))

if notebooks:
    nb = nbformat.read(notebooks[-1], as_version=4)

    print("\nNotebook cells:")
    for i, cell in enumerate(nb.cells):
        if cell.cell_type == "code":
            print(f"\nCell {i}:")
            print(cell.source[:180].replace("\n", " "))

Notebooks found:
0 Untitled16.ipynb

Notebook cells:

Cell 0:
""" Part A - Data Foundation Tata Play Subscriber Retention & Support Copilot  Tasks: 1. Audit raw datasets 2. Clean subscribers 3. Clean recharges 4. Save clean CSV files 5. Creat

Cell 1:
 from pathlib import Path import warnings  import joblib import numpy as np import pandas as pd  from sklearn.compose import ColumnTransformer from sklearn.impute import SimpleImpu

Cell 2:
 # ============================================================ # PART C: SEGMENTS, SPEND MODEL AND REVENUE AT RISK # Tata Play Subscriber Retention & Support Copilot # ===========

Cell 3:
from google.colab import files uploaded = files.upload()

Cell 4:
import sys sys.path.insert(0, "/content")  import mock_llm print("mock_llm imported successfully!")

Cell 5:
!pip install -q "pydantic>=2,<3" langchain-core chromadb sentence-transformers

Cell 6:
 %%writefile /content/llm_client.py  import os import time import mock_llm  BASE_DELAY = 0.05 REAL_URL = 

In [55]:
import nbformat
from pathlib import Path

notebook = nbformat.read("/content/Untitled16.ipynb", as_version=4)

parts = {
    "part_a.py": 0,
    "part_b.py": 1,
    "part_c.py": 2
}

for filename, cell_number in parts.items():
    source = notebook.cells[cell_number].source
    Path(f"/content/{filename}").write_text(source, encoding="utf-8")
    print(f"Created: {filename} ({len(source)} characters)")

print("\nAll three Python files created!")

Created: part_a.py (15587 characters)
Created: part_b.py (27084 characters)
Created: part_c.py (16410 characters)

All three Python files created!


In [56]:
import zipfile
from google.colab import files

names = ["part_a.py", "part_b.py", "part_c.py"]

with zipfile.ZipFile("Missing_Python_Files.zip", "w") as z:
    for name in names:
        z.write(name)

files.download("Missing_Python_Files.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
!git clone https://github.com/preetham-10/Tata-Play-Retention-Copilot.git

Cloning into 'Tata-Play-Retention-Copilot'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 60 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 863.41 KiB | 4.77 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [58]:
%cd /content/Tata-Play-Retention-Copilot
!pip install -r requirements.txt

/content/Tata-Play-Retention-Copilot
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.3 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.

In [1]:
import os

print("Project files:")
print(os.listdir("/content/Tata-Play-Retention-Copilot"))

Project files:
['policy_eval.csv', 'part_d.py', 'q3_ranking_within_city.csv', 'README.md', 'retention_priority.csv', 'q4_running_total.csv', 'random_forest_feature_importance.csv', 'ticket_extractions.jsonl', 'plans.csv', 'tataplay.db', 'Missing_Python_Files.zip', 'retention_briefs.jsonl', 'logistic_thresholds.csv', 'requirements.txt', 'logistic_regression_coefficients.csv', 'copilot.py', 'q1_revenue_by_plan.csv', 'retention_offers_policy.txt', 'recharges.csv', 'forest_thresholds.csv', 'queries.sql', '.git', 'mock_llm.py', 'spend_model.joblib', '.gitignore', 'llm_client.py', 'cancellation_policy.txt', 'tickets.csv', 'decision_tree_depth.csv', 'technical_support_policy.txt', 'ticket_classification.csv', 'plan_change_policy.txt', 'recharge_and_refund_policy.txt', 'subscribers.csv', 'test_scores.csv', 'q2_never_successfully_recharged.csv', 'part_a.py', 'part_b.py', 'generate_data.py', 'rag_eval.csv', 'segments.csv', 'few_shot_examples.csv', 'part_c.py', 'churn_model.joblib']


In [2]:
!python part_a.py

<IPython.core.display.HTML object>
Traceback (most recent call last):
  File "/content/part_a.py", line 18, in <module>
    uploaded = files.upload()
  File "/usr/local/lib/python3.13/dist-packages/google/colab/files.py", line 69, in upload
    uploaded_files = _upload_files(multiple=True)
  File "/usr/local/lib/python3.13/dist-packages/google/colab/files.py", line 161, in _upload_files
    result = _output.eval_js(
        'google.colab._files._uploadFiles("{input_id}", "{output_id}")'.format(
            input_id=input_id, output_id=output_id
        )
    )
  File "/usr/local/lib/python3.13/dist-packages/google/colab/output/_js.py", line 36, in eval_js
    kernel = _ipython.get_kernel()
  File "/usr/local/lib/python3.13/dist-packages/google/colab/_ipython.py", line 24, in get_kernel
    return get_ipython().kernel
           ^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'kernel'


In [4]:
%cd /content/Tata-Play-Retention-Copilot
%run part_a.py

/content/Tata-Play-Retention-Copilot


Saving tataplay_colab_dataset.zip to tataplay_colab_dataset.zip
Files extracted successfully!
/content/Tata-Play-Retention-Copilot/subscribers.csv
/content/data/raw/subscribers.csv
/content/data/clean/subscribers.csv

========== PART A: DATA FOUNDATION ==========


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/subscribers.csv'

In [5]:
import os

print("Current directory:", os.getcwd())

print("\nExtracted data files:")
for root, dirs, files in os.walk("/content/data"):
    for file in files:
        print(os.path.join(root, file))

Current directory: /content/Tata-Play-Retention-Copilot

Extracted data files:
/content/data/tataplay.db
/content/data/policies/retention_offers_policy.txt
/content/data/policies/cancellation_policy.txt
/content/data/policies/technical_support_policy.txt
/content/data/policies/plan_change_policy.txt
/content/data/policies/recharge_and_refund_policy.txt
/content/data/raw/policy_eval.csv
/content/data/raw/plans.csv
/content/data/raw/recharges.csv
/content/data/raw/tickets.csv
/content/data/raw/subscribers.csv
/content/data/raw/few_shot_examples.csv
/content/data/clean/plans.csv
/content/data/clean/recharges.csv
/content/data/clean/subscribers.csv


In [6]:
import shutil
import os

source = "/content/data"
destination = "/content/Tata-Play-Retention-Copilot/data"

if os.path.exists(source):
    shutil.copytree(source, destination, dirs_exist_ok=True)

print("Data folders copied successfully!")

Data folders copied successfully!


In [7]:
%cd /content/Tata-Play-Retention-Copilot
%run part_a.py

/content/Tata-Play-Retention-Copilot


Saving tataplay_colab_dataset.zip to tataplay_colab_dataset (1).zip
Files extracted successfully!
/content/Tata-Play-Retention-Copilot/subscribers.csv
/content/Tata-Play-Retention-Copilot/data/raw/subscribers.csv
/content/Tata-Play-Retention-Copilot/data/clean/subscribers.csv
/content/data/raw/subscribers.csv
/content/data/clean/subscribers.csv

========== PART A: DATA FOUNDATION ==========

Raw datasets loaded successfully.

========== SUBSCRIBERS AUDIT ==========

Shape:
(2415, 17)

Data types:
subscriber_id               object
city                        object
plan_id                     object
tenure_months                int64
age                        float64
payment_mode                object
auto_renew                  object
add_on_count                 int64
monthly_recharge_inr         int64
hours_watched_per_week     float64
hd_hours_per_week          float64
sports_hours_per_week      float64
channels_watched             int64
support_calls_90d            int64
closure_

In [8]:
%cd /content/Tata-Play-Retention-Copilot
%run part_b.py

/content/Tata-Play-Retention-Copilot

========== PART B: CHURN PREDICTION ==========
Subscribers: (2400, 17)
Recharges: (12985, 6)

Recharge feature checks:
Missing recency_days: 76
Total successful recharges in 6 months: 11442
Total failed recharges in 6 months: 1422

========== CLASS BALANCE ==========
Stayed (0): 1886
Churned (1): 514
Churn percentage: 21.42 %

========== DATA SPLIT ==========
Training: (1440, 16) | churners: 308
Validation: (480, 16) | churners: 103
Test: (480, 16) | churners: 103

Training is used to fit preprocessing and models; validation is used for model/threshold selection; test is used once for final evaluation.

========== PREPROCESSING ==========
Output feature count: 31
Missing values after transform: 0
Learned median age: 38.0
Scaler mean of tenure_months: 22.65277777777778
Full-data mean of tenure_months: 22.46

========== DATA LEAKAGE EXPERIMENT ==========
WITH leakage:
Accuracy: 0.983
Churn F1: 0.961

WITHOUT leakage:
Accuracy: 0.823
Churn F1: 0.514



In [9]:
%cd /content/Tata-Play-Retention-Copilot
%run part_c.py

/content/Tata-Play-Retention-Copilot
Subscribers: (2400, 17)
Part B test scores: (480, 3)

========== K-MEANS CLUSTERING ==========

K-means evaluation:
 K  Inertia  Silhouette Score
 2 3728.219             0.420
 3 2812.958             0.346
 4 2193.933             0.340
 5 1781.092             0.343
 6 1502.476             0.344
 7 1292.830             0.364
 8 1191.417             0.336

========== SEGMENT PROFILES ==========
 segment_id  subscriber_count  mean_tenure  mean_weekly_hours  mean_monthly_recharge  churn_rate
          0               593        23.58               7.08                 301.33       22.43
          1               543        34.16              22.24                 559.73       13.44
          2               480        34.20              33.64                 942.32       10.62
          3               784         6.32              14.85                 513.51       32.78

Named segment profile:
 segment_id segment_name  subscriber_count  mean_tenure  m

In [10]:
%cd /content/Tata-Play-Retention-Copilot
%run part_d.py

/content/Tata-Play-Retention-Copilot
Ticket classification completed.
  ticket_id subscriber_id                                        ticket_text  \
0     T0001       TP02169  Subscriber ID TP02169. Please cancel my subscr...   
1     T0002       TP01419  Subscriber ID TP01419. The picture is fine but...   
2     T0003       TP01581  I am closing the account because the price is ...   
3     T0004       TP01501  Subscriber ID TP01501. There is no signal on m...   
4     T0005       TP02291  Auto renew took money from my account but the ...   

  true_category zero_shot_prediction few_shot_prediction  \
0  cancellation         cancellation        cancellation   
1       billing              billing             billing   
2  cancellation         cancellation        cancellation   
3        signal               signal              signal   
4       billing              billing             billing   

                                weak_prompt_response  manual_review  
0  This looks like

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG evaluation completed.
                                            question  \
0  My top-up money was deducted but the recharge ...   
1  I paid cash at a shop and need my money return...   
2  I topped up the wrong customer ID by mistake. ...   
3  What happens if my automatic payment does not ...   
4         Can I switch to a cheaper pack right away?   

             expected_policy_file  \
0  recharge_and_refund_policy.txt   
1  recharge_and_refund_policy.txt   
2  recharge_and_refund_policy.txt   
3  recharge_and_refund_policy.txt   
4          plan_change_policy.txt   

                              retrieved_policy_files  \
0  recharge_and_refund_policy.txt; cancellation_p...   
1  recharge_and_refund_policy.txt; cancellation_p...   
2  recharge_and_refund_policy.txt; cancellation_p...   
3  recharge_and_refund_policy.txt; cancellation_p...   
4  plan_change_policy.txt; retention_offers_polic...   

                                              answer       expected_fact  \
0

In [11]:
%cd /content/Tata-Play-Retention-Copilot
%run copilot.py

/content/Tata-Play-Retention-Copilot
Retention briefs generated successfully!
Total subscribers: 15
{
  "subscriber_id": "TP00798",
  "risk_band": "High",
  "recommended_action": "Offer an auto-renew setup call and a 7 day grace extension.",
  "policy_source": "retention_offers_policy.txt",
  "tenure_months": 13,
  "failed_recharges_6m": 3,
  "churn_probability": 0.8176170892177579
}
